# Lab Day 1 — Neural network and Training

Notebook đã hoàn thiện các phần Part 0–4, gồm các module code, thí nghiệm validation và xuất sản phẩm nộp.

**Chạy trên Kaggle:** bật Internet và GPU trong Notebook settings rồi chọn Run All. Ô đầu tiên clone repository công khai để lấy dữ liệu cố định/script/template; các module đã hoàn thiện được đóng gói cùng notebook. Kết quả được tạo dưới `submission_working/` trong `/kaggle/working/`.

**Giới hạn lựa chọn:** chỉ dùng validation để chọn learning rate, epoch và cấu hình. Eval chỉ chạy cuối cho baseline và mô hình cuối. Điền MSSV thật vào tên thư mục trước khi nộp.


In [ ]:
# Kaggle bootstrap: clone the fixed lab assets, then install the completed local modules.
import base64, os, subprocess
from pathlib import Path
_repo_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
_repo_found = next((p.resolve() for p in _repo_candidates if (p / 'scripts' / 'split_data.py').exists()), None)
if _repo_found is None and Path('/kaggle/working').exists():
    _repo_found = Path('/kaggle/working/K4-Track4-Day1-Neuralnetwork')
    if not (_repo_found / 'scripts' / 'split_data.py').exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/VinUni-AI20k/K4-Track4-Day1-Neuralnetwork.git',str(_repo_found)],check=True)
    _bundle = {"data.py":"IiIiUGFydCAwIGRhdGEgbG9hZGluZywgdmFsaWRhdGlvbiBzcGxpdHRpbmcsIGFuZCBzdGFuZGFyZGl6YXRpb24uDQoNCk5oaeG7h20gduG7pTogbuG6oXAgdOG6rXAgdHJhaW4vZXZhbCDEkcOjIGNoaWEgc+G6tW4sIHTDoWNoIHZhbGlkYXRpb24gdOG7qyB0cmFpbiwgY2h14bqpbiBob8OhLCDEkcawYSBsw6puIHRoaeG6v3QgYuG7iy4NCg0KxJBp4buBdSBraeG7h24gdHLGsOG7m2M6IMSRw6MgY2jhuqF5IGBweXRob24gc2NyaXB0cy9zcGxpdF9kYXRhLnB5YCAodOG6oW8gZGF0YS9wcm9jZXNzZWQvdHJhaW4ubnB6LCBldmFsLm5weikuDQoNClF1eSDGsOG7m2MgZOG7ryBsaeG7h3UgKHhlbSBSRUFETUUgbeG7pWMgMiB2w6AgMyk6DQogICAgWCA6IGZsb2F0MzIsIHNoYXBlIChOLCA1NCkgICDigJQgMTAgY+G7mXQgxJHhuqd1IGzDoCBz4buRIGxpw6puIHThu6VjLCA0NCBj4buZdCBzYXUgbMOgIG5o4buLIHBow6JuIChvbmUtaG90KQ0KICAgIHkgOiBpbnQ2NCwgICBzaGFwZSAoTiwpICAgICAg4oCUIG5ow6NuIDAuLjYNClThuq1wIGV2YWwgQ0jhu4ggZMO5bmcgxJHhu4MgY2jhuqVtIMSRaeG7g20gY3Xhu5FpLiBLaMO0bmcgZMO5bmcgbsOzIMSR4buDIGNo4buNbiBj4bqldSBow6xuaCwgY2h14bqpbiBob8OhIGhheSBk4burbmcgc+G7m20uDQoiIiINCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgdG9yY2gNCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQNCg0KTl9OVU1FUklDID0gMTAgICMgc+G7kSBj4buZdCBsacOqbiB04bulYyBj4bqnbiBjaHXhuqluIGhvw6EgKGPhu5l0IDAuLjkpDQoNCg0KZGVmIGxvYWRfc3BsaXQocHJvY2Vzc2VkX2Rpcjogc3RyID0gImRhdGEvcHJvY2Vzc2VkIik6DQogICAgIiIiTuG6oXAgdHJhaW4gdsOgIGV2YWwgdOG7qyBmaWxlIC5ucHouDQoNCiAgICBUcuG6oyB24buBOiBYX3RyYWluX2Z1bGwsIHlfdHJhaW5fZnVsbCwgWF9ldmFsLCB5X2V2YWwsIGV2YWxfcm93X2lkDQogICAgQ8OhYyBixrDhu5tjOg0KICAgICAgMS4gbnAubG9hZChmIntwcm9jZXNzZWRfZGlyfS90cmFpbi5ucHoiKSAtPiBraG/DoSAiWCIsICJ5Ig0KICAgICAgMi4gbnAubG9hZChmIntwcm9jZXNzZWRfZGlyfS9ldmFsLm5weiIpICAtPiBraG/DoSAiWCIsICJ5IiwgInJvd19pZCINCiAgICAgIDMuIGFzc2VydCBzaGFwZS9kdHlwZSDEkcO6bmcgcXV5IMaw4bubYyDhu58gxJHhuqd1IGZpbGUNCiAgICAiIiINCiAgICB0cmFpbl9wYXRoID0gZiJ7cHJvY2Vzc2VkX2Rpcn0vdHJhaW4ubnB6Ig0KICAgIGV2YWxfcGF0aCA9IGYie3Byb2Nlc3NlZF9kaXJ9L2V2YWwubnB6Ig0KICAgIHdpdGggbnAubG9hZCh0cmFpbl9wYXRoKSBhcyB0cmFpbiwgbnAubG9hZChldmFsX3BhdGgpIGFzIGV2YWx1YXRpb246DQogICAgICAgIFhfdHJhaW4gPSB0cmFpblsiWCJdDQogICAgICAgIHlfdHJhaW4gPSB0cmFpblsieSJdDQogICAgICAgIFhfZXZhbCA9IGV2YWx1YXRpb25bIlgiXQ0KICAgICAgICB5X2V2YWwgPSBldmFsdWF0aW9uWyJ5Il0NCiAgICAgICAgZXZhbF9yb3dfaWQgPSBldmFsdWF0aW9uWyJyb3dfaWQiXQ0KDQogICAgYXNzZXJ0IFhfdHJhaW4ubmRpbSA9PSBYX2V2YWwubmRpbSA9PSAyIGFuZCBYX3RyYWluLnNoYXBlWzFdID09IFhfZXZhbC5zaGFwZVsxXSA9PSA1NA0KICAgIGFzc2VydCBYX3RyYWluLmR0eXBlID09IFhfZXZhbC5kdHlwZSA9PSBucC5mbG9hdDMyDQogICAgYXNzZXJ0IHlfdHJhaW4uc2hhcGUgPT0gKGxlbihYX3RyYWluKSwpIGFuZCB5X2V2YWwuc2hhcGUgPT0gKGxlbihYX2V2YWwpLCkNCiAgICBhc3NlcnQgeV90cmFpbi5kdHlwZSA9PSB5X2V2YWwuZHR5cGUgPT0gbnAuaW50NjQNCiAgICBhc3NlcnQgZXZhbF9yb3dfaWQuc2hhcGUgPT0gKGxlbihYX2V2YWwpLCkgYW5kIGV2YWxfcm93X2lkLmR0eXBlID09IG5wLmludDY0DQogICAgYXNzZXJ0IG5wLmlzZmluaXRlKFhfdHJhaW4pLmFsbCgpIGFuZCBucC5pc2Zpbml0ZShYX2V2YWwpLmFsbCgpDQogICAgYXNzZXJ0IHlfdHJhaW4ubWluKCkgPj0gMCBhbmQgeV90cmFpbi5tYXgoKSA8PSA2DQogICAgYXNzZXJ0IHlfZXZhbC5taW4oKSA+PSAwIGFuZCB5X2V2YWwubWF4KCkgPD0gNg0KICAgIHJldHVybiBYX3RyYWluLCB5X3RyYWluLCBYX2V2YWwsIHlfZXZhbCwgZXZhbF9yb3dfaWQNCg0KDQpkZWYgbWFrZV92YWxfc3BsaXQoWCwgeSwgdmFsX2ZyYWN0aW9uOiBmbG9hdCA9IDAuMiwgc2VlZDogaW50ID0gNDIpOg0KICAgICIiIlTDoWNoIHZhbGlkYXRpb24gVOG7qiB0cmFpbiAoa2jDtG5nIMSR4bulbmcgZXZhbCkuIFBow6JuIHThuqduZyB0aGVvIG5ow6NuLg0KDQogICAgVHLhuqMgduG7gTogWF90ciwgeV90ciwgWF92YWwsIHlfdmFsDQogICAgR+G7o2kgw706IHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uLnRyYWluX3Rlc3Rfc3BsaXQoLi4uLCBzdHJhdGlmeT15LCByYW5kb21fc3RhdGU9c2VlZCkNCiAgICBEw7luZyBDw5lORyBzZWVkIHbDoCB2YWxfZnJhY3Rpb24gY2hvIG3hu41pIHRow60gbmdoaeG7h20gxJHhu4Mgc28gc8OhbmggY8O0bmcgYuG6sW5nLg0KICAgICIiIg0KICAgIGlmIG5vdCAwLjAgPCB2YWxfZnJhY3Rpb24gPCAxLjA6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInZhbF9mcmFjdGlvbiBtdXN0IGJlIGJldHdlZW4gMCBhbmQgMSIpDQogICAgWF90ciwgWF92YWwsIHlfdHIsIHlfdmFsID0gdHJhaW5fdGVzdF9zcGxpdCgNCiAgICAgICAgWCwgeSwgdGVzdF9zaXplPXZhbF9mcmFjdGlvbiwgcmFuZG9tX3N0YXRlPXNlZWQsIHN0cmF0aWZ5PXkNCiAgICApDQogICAgcmV0dXJuIFhfdHIsIHlfdHIsIFhfdmFsLCB5X3ZhbA0KDQoNCmRlZiBmaXRfc3RhbmRhcmRpemVyKFhfdHIpOg0KICAgICIiIlTDrW5oIG1lYW4gdsOgIHN0ZCBj4bunYSBOX05VTUVSSUMgY+G7mXQgxJHhuqd1IENI4buIIHRyw6puIHThuq1wIHRyYWluIChzYXUga2hpIHTDoWNoIHZhbCkuDQoNCiAgICBUcuG6oyB24buBOiBtZWFuIChzaGFwZSAoMTAsKSksIHN0ZCAoc2hhcGUgKDEwLCkpDQogICAgQ8OidSBo4buPaTogdsOsIHNhbyBraMO0bmcgxJHGsOG7o2MgdMOtbmggdHLDqm4gdG/DoG4gYuG7mSBk4buvIGxp4buHdSBoYXkgdHLDqm4gZXZhbD8NCiAgICAiIiINCiAgICBYX3RyID0gbnAuYXNhcnJheShYX3RyLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgIGlmIFhfdHIubmRpbSAhPSAyIG9yIFhfdHIuc2hhcGVbMV0gPCBOX05VTUVSSUM6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJYX3RyIG11c3QgaGF2ZSBhdCBsZWFzdCB7Tl9OVU1FUklDfSBmZWF0dXJlIGNvbHVtbnMiKQ0KICAgIG1lYW4gPSBYX3RyWzosIDpOX05VTUVSSUNdLm1lYW4oYXhpcz0wLCBkdHlwZT1ucC5mbG9hdDY0KS5hc3R5cGUobnAuZmxvYXQzMikNCiAgICBzdGQgPSBYX3RyWzosIDpOX05VTUVSSUNdLnN0ZChheGlzPTAsIGR0eXBlPW5wLmZsb2F0NjQpLmFzdHlwZShucC5mbG9hdDMyKQ0KICAgICMgQSBjb25zdGFudCBmZWF0dXJlIG5lZWRzIG5vIHJlc2NhbGluZzsgdXNpbmcgMSBhdm9pZHMgZGl2aXNpb24gYnkgemVyby4NCiAgICBzdGRbc3RkID09IDBdID0gMS4wDQogICAgcmV0dXJuIG1lYW4sIHN0ZA0KDQoNCmRlZiBhcHBseV9zdGFuZGFyZGl6ZXIoWCwgbWVhbiwgc3RkKToNCiAgICAiIiJUcuG6oyB24buBIGLhuqNuIHNhbyBj4bunYSBYLCB0cm9uZyDEkcOzIDEwIGPhu5l0IMSR4bqndSDEkcaw4bujYyAoeCAtIG1lYW4pIC8gc3RkOyA0NCBj4buZdCBuaOG7iyBwaMOibiBnaeG7ryBuZ3V5w6puLg0KDQogICAgQ2jDuiDDvToga2jDtG5nIHPhu61hIFggdOG6oWkgY2jhu5cgbuG6v3UgYuG6oW4gY8OybiBkw7luZyBs4bqhaSBuw7M7IGNow7ogw70gc3RkID0gMCAobuG6v3UgY8OzKS4NCiAgICAiIiINCiAgICBYID0gbnAuYXNhcnJheShYLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgIGlmIFgubmRpbSAhPSAyIG9yIFguc2hhcGVbMV0gPCBOX05VTUVSSUM6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJYIG11c3QgaGF2ZSBhdCBsZWFzdCB7Tl9OVU1FUklDfSBmZWF0dXJlIGNvbHVtbnMiKQ0KICAgIG1lYW4gPSBucC5hc2FycmF5KG1lYW4sIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgc3RkID0gbnAuYXNhcnJheShzdGQsIGR0eXBlPW5wLmZsb2F0MzIpDQogICAgaWYgbWVhbi5zaGFwZSAhPSAoTl9OVU1FUklDLCkgb3Igc3RkLnNoYXBlICE9IChOX05VTUVSSUMsKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIm1lYW4gYW5kIHN0ZCBtdXN0IGJvdGggaGF2ZSBzaGFwZSAoe05fTlVNRVJJQ30sKSIpDQogICAgaWYgbnAuYW55KHN0ZCA8PSAwKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic3RkIHZhbHVlcyBtdXN0IGJlIHBvc2l0aXZlOyB1c2UgZml0X3N0YW5kYXJkaXplciB0byBoYW5kbGUgY29uc3RhbnRzIikNCiAgICB0cmFuc2Zvcm1lZCA9IFguY29weSgpDQogICAgdHJhbnNmb3JtZWRbOiwgOk5fTlVNRVJJQ10gPSAodHJhbnNmb3JtZWRbOiwgOk5fTlVNRVJJQ10gLSBtZWFuKSAvIHN0ZA0KICAgIHJldHVybiB0cmFuc2Zvcm1lZA0KDQoNCmRlZiBwcmVwYXJlX2RhdGEoZGV2aWNlOiBzdHIsIHZhbF9mcmFjdGlvbjogZmxvYXQgPSAwLjIsIHNlZWQ6IGludCA9IDQyLA0KICAgICAgICAgICAgICAgICBwcm9jZXNzZWRfZGlyOiBzdHIgPSAiZGF0YS9wcm9jZXNzZWQiKSAtPiBkaWN0Og0KICAgICIiIkfhu5lwIGPDoWMgYsaw4bubYyB0csOqbiB2w6AgxJHGsGEgVE/DgE4gQuG7mCBk4buvIGxp4buHdSBsw6puIGBkZXZpY2VgIG3hu5l0IGzhuqduIChraMO0bmcgZMO5bmcgRGF0YUxvYWRlcikuDQoNCiAgICBUcuG6oyB24buBIGRpY3QgZ+G7k20gY8OhYyB0ZW5zb3IgdHLDqm4gZGV2aWNlOg0KICAgICAgICBYX3RyLCB5X3RyLCBYX3ZhbCwgeV92YWwsIFhfZXZhbCwgeV9ldmFsICAgICAgICAoeSBsw6AgaW50NjQpDQogICAgdsOgIGPDoWMgbeG6o25nIG51bXB5OiBldmFsX3Jvd19pZA0KICAgIEPDoWMgYsaw4bubYzoNCiAgICAgIDEuIGxvYWRfc3BsaXQgLT4gbWFrZV92YWxfc3BsaXQgLT4gZml0X3N0YW5kYXJkaXplciAoY2jhu4kgdHLDqm4gWF90cikNCiAgICAgIDIuIGFwcGx5X3N0YW5kYXJkaXplciBjaG8gWF90ciwgWF92YWwsIFhfZXZhbCBi4bqxbmcgQ8OZTkcgbWVhbi9zdGQNCiAgICAgIDMuIHRvcmNoLnRlbnNvciguLi4sIGRldmljZT1kZXZpY2UpOyBYIGzDoCBmbG9hdDMyLCB5IGzDoCBpbnQ2NA0KICAgICAgNC4gaW4gcmEga8OtY2ggdGjGsOG7m2MgY8OhYyB04bqtcCB2w6AgYWNjdXJhY3kgY+G7p2EgY2hp4bq/biBsxrDhu6NjICJsdcO0biDEkW/DoW4gbOG7m3AgxJFhIHPhu5EiIHRyw6puIHZhbA0KICAgICIiIg0KICAgIFhfdHJhaW5fZnVsbCwgeV90cmFpbl9mdWxsLCBYX2V2YWwsIHlfZXZhbCwgZXZhbF9yb3dfaWQgPSBsb2FkX3NwbGl0KHByb2Nlc3NlZF9kaXIpDQogICAgWF90ciwgeV90ciwgWF92YWwsIHlfdmFsID0gbWFrZV92YWxfc3BsaXQoDQogICAgICAgIFhfdHJhaW5fZnVsbCwgeV90cmFpbl9mdWxsLCB2YWxfZnJhY3Rpb249dmFsX2ZyYWN0aW9uLCBzZWVkPXNlZWQNCiAgICApDQogICAgbWVhbiwgc3RkID0gZml0X3N0YW5kYXJkaXplcihYX3RyKQ0KICAgIFhfdHIgPSBhcHBseV9zdGFuZGFyZGl6ZXIoWF90ciwgbWVhbiwgc3RkKQ0KICAgIFhfdmFsID0gYXBwbHlfc3RhbmRhcmRpemVyKFhfdmFsLCBtZWFuLCBzdGQpDQogICAgWF9ldmFsID0gYXBwbHlfc3RhbmRhcmRpemVyKFhfZXZhbCwgbWVhbiwgc3RkKQ0KDQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKGRldmljZSkNCiAgICBkYXRhID0gew0KICAgICAgICAiWF90ciI6IHRvcmNoLmFzX3RlbnNvcihYX3RyLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9ZGV2aWNlKSwNCiAgICAgICAgInlfdHIiOiB0b3JjaC5hc190ZW5zb3IoeV90ciwgZHR5cGU9dG9yY2guaW50NjQsIGRldmljZT1kZXZpY2UpLA0KICAgICAgICAiWF92YWwiOiB0b3JjaC5hc190ZW5zb3IoWF92YWwsIGR0eXBlPXRvcmNoLmZsb2F0MzIsIGRldmljZT1kZXZpY2UpLA0KICAgICAgICAieV92YWwiOiB0b3JjaC5hc190ZW5zb3IoeV92YWwsIGR0eXBlPXRvcmNoLmludDY0LCBkZXZpY2U9ZGV2aWNlKSwNCiAgICAgICAgIlhfZXZhbCI6IHRvcmNoLmFzX3RlbnNvcihYX2V2YWwsIGR0eXBlPXRvcmNoLmZsb2F0MzIsIGRldmljZT1kZXZpY2UpLA0KICAgICAgICAieV9ldmFsIjogdG9yY2guYXNfdGVuc29yKHlfZXZhbCwgZHR5cGU9dG9yY2guaW50NjQsIGRldmljZT1kZXZpY2UpLA0KICAgICAgICAiZXZhbF9yb3dfaWQiOiBldmFsX3Jvd19pZCwNCiAgICAgICAgInN0YW5kYXJkaXplcl9tZWFuIjogbWVhbiwNCiAgICAgICAgInN0YW5kYXJkaXplcl9zdGQiOiBzdGQsDQogICAgfQ0KDQogICAgbWFqb3JpdHlfbGFiZWwgPSBpbnQobnAuYmluY291bnQoeV90ciwgbWlubGVuZ3RoPTcpLmFyZ21heCgpKQ0KICAgIHZhbF9tYWpvcml0eV9hY2N1cmFjeSA9IGZsb2F0KG5wLm1lYW4oeV92YWwgPT0gbWFqb3JpdHlfbGFiZWwpKQ0KICAgIHByaW50KGYidHJhaW46IHt0dXBsZShkYXRhWydYX3RyJ10uc2hhcGUpfSwgdmFsOiB7dHVwbGUoZGF0YVsnWF92YWwnXS5zaGFwZSl9LCBldmFsOiB7dHVwbGUoZGF0YVsnWF9ldmFsJ10uc2hhcGUpfSIpDQogICAgcHJpbnQoZiJkZXZpY2U6IHtkZXZpY2V9OyBtYWpvcml0eSBjbGFzcyBmcm9tIHRyYWluOiB7bWFqb3JpdHlfbGFiZWx9OyB2YWwgbWFqb3JpdHkgYWNjdXJhY3k6IHt2YWxfbWFqb3JpdHlfYWNjdXJhY3k6LjRmfSIpDQogICAgcHJpbnQoZiJzdGFuZGFyZGl6ZXIgdHJhaW4gbWVhbiAoMTAgbnVtZXJpYyBmZWF0dXJlcyk6IHtkYXRhWydYX3RyJ11bOiwgOk5fTlVNRVJJQ10ubWVhbihkaW09MCkuZGV0YWNoKCkuY3B1KCkubnVtcHkoKS5yb3VuZCg0KX0iKQ0KICAgIHByaW50KGYic3RhbmRhcmRpemVyIHRyYWluIHN0ZCAgKDEwIG51bWVyaWMgZmVhdHVyZXMpOiB7ZGF0YVsnWF90ciddWzosIDpOX05VTUVSSUNdLnN0ZChkaW09MCwgdW5iaWFzZWQ9RmFsc2UpLmRldGFjaCgpLmNwdSgpLm51bXB5KCkucm91bmQoNCl9IikNCiAgICByZXR1cm4gZGF0YQ0KDQoNCmRlZiBpdGVyYXRlX2JhdGNoZXMoWCwgeSwgYmF0Y2hfc2l6ZTogaW50LCBnZW5lcmF0b3I6IHRvcmNoLkdlbmVyYXRvciB8IE5vbmUgPSBOb25lLCBzaHVmZmxlOiBib29sID0gVHJ1ZSk6DQogICAgIiIiR2VuZXJhdG9yIHRy4bqjIHbhu4EgdOG7q25nIGPhurdwICh4YiwgeWIpLCB0aGF5IGNobyBEYXRhTG9hZGVyLg0KDQogICAgQ8OhYyBixrDhu5tjOg0KICAgICAgMS4gbuG6v3Ugc2h1ZmZsZTogcGVybSA9IHRvcmNoLnJhbmRwZXJtKGxlbihYKSwgZ2VuZXJhdG9yPWdlbmVyYXRvciwgZGV2aWNlPVguZGV2aWNlKTsgbmfGsOG7o2MgbOG6oWkgYXJhbmdlDQogICAgICAyLiBmb3IgaSBpbiByYW5nZSgwLCBOLCBiYXRjaF9zaXplKTogaWR4ID0gcGVybVtpOmkrYmF0Y2hfc2l6ZV07IHlpZWxkIFhbaWR4XSwgeVtpZHhdDQogICAgQ2jDuiDDvTogYmF0Y2ggY3Xhu5FpIGPDsyB0aOG7gyBuaOG7jyBoxqFuIGJhdGNoX3NpemU7IGjDo3kgcXV54bq/dCDEkeG7i25oIGLhuqFuIHjhu60gbMO9IHRo4bq/IG7DoG8gdsOgIGdoaSBs4bqhaS4NCiAgICAiIiINCiAgICBpZiBiYXRjaF9zaXplIDw9IDA6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoX3NpemUgbXVzdCBiZSBwb3NpdGl2ZSIpDQogICAgaWYgbGVuKFgpICE9IGxlbih5KToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiWCBhbmQgeSBtdXN0IGNvbnRhaW4gdGhlIHNhbWUgbnVtYmVyIG9mIHNhbXBsZXMiKQ0KICAgIG5fc2FtcGxlcyA9IGxlbihYKQ0KICAgIGlmIHNodWZmbGU6DQogICAgICAgIHBlcm0gPSB0b3JjaC5yYW5kcGVybShuX3NhbXBsZXMsIGdlbmVyYXRvcj1nZW5lcmF0b3IsIGRldmljZT1YLmRldmljZSkNCiAgICBlbHNlOg0KICAgICAgICBwZXJtID0gdG9yY2guYXJhbmdlKG5fc2FtcGxlcywgZGV2aWNlPVguZGV2aWNlKQ0KICAgIGZvciBzdGFydCBpbiByYW5nZSgwLCBuX3NhbXBsZXMsIGJhdGNoX3NpemUpOg0KICAgICAgICBpbmRpY2VzID0gcGVybVtzdGFydCA6IHN0YXJ0ICsgYmF0Y2hfc2l6ZV0NCiAgICAgICAgeWllbGQgWFtpbmRpY2VzXSwgeVtpbmRpY2VzXQ0K","model.py":"IiIiTUxQIG1vZGVscyBhbmQgaW5pdGlhbGl6YXRpb24gaGVscGVycyBmb3IgdGhlIG5ldXJhbC1uZXR3b3JrIGxhYi4NCg0KTW9kZWw6IE1MUCBjaG8gYsOgaSB0b8OhbiA3IGzhu5twLCBzaGFwZSBj4buRIMSR4buLbmggKHhlbSBSRUFETUUgbeG7pWMgMyB2w6AgR1VJREUsICJRdXkgxJHhu4tuaCBraeG6v24gdHLDumMiKToNCg0KICAgIHggKEIsIDU0KSAtPiBMaW5lYXIoNTQsIGgxKSAtPiBSZUxVIC0+IFtEcm9wb3V0XSAtPiBMaW5lYXIoaDEsIGgyKSAtPiBSZUxVIC0+IFtEcm9wb3V0XQ0KICAgICAgICAgICAgICAtPiAuLi4gLT4gTGluZWFyKGhfbGFzdCwgNykgLT4gbG9naXRzIChCLCA3KQ0KDQpRdXkgdOG6r2M6DQogIC0gTOG7m3AgY3Xhu5FpIHJhIGxvZ2l0IHRow7QsIEtIw5RORyBzb2Z0bWF4IHRyb25nIG1vZGVsIChzb2Z0bWF4IG7hurFtIHRyb25nIGjDoG0gbeG6pXQgbcOhdCkuDQogIC0gRHJvcG91dCBjaOG7iSDEkeG6t3Qgc2F1IFJlTFUgY+G7p2EgbOG7m3Ag4bqpbjsga2jDtG5nIMSR4bq3dCB0csOqbiDEkeG6p3UgdsOgbyBoYXkgbG9naXQuDQogIC0gTeG7jWkgbm4uTGluZWFyIMSR4buBdSBjw7MgYmlhcy4gS2jDtG5nIEJhdGNoTm9ybSwga2jDtG5nIHJlc2lkdWFsLg0KICAtIFPhu5EgdGhhbSBz4buRIHBo4bqjaSBraOG7m3AgRVhQRUNURURfUEFSQU1TIGLDqm4gZMaw4bubaS4NCiIiIg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgdG9yY2gNCmltcG9ydCB0b3JjaC5ubiBhcyBubg0KDQojIFPhu5EgdGhhbSBz4buRIGLhuq90IGJ14buZYyDhu6luZyB24bubaSB04burbmcga2nhur9uIHRyw7pjIChpbl9mZWF0dXJlcz01NCwgbnVtX2NsYXNzZXM9NykNCkVYUEVDVEVEX1BBUkFNUyA9IHsNCiAgICAoMjU2LCAxMjgpOiA0N184NzksICAgICAgICAjIE0tYmFzZSAgKGJhc2VsaW5lKQ0KICAgICg1MTIsIDI1Nik6IDE2MV8yODcsICAgICAgICMgTS13aWRlICAodHXhu7MgY2jhu41uKQ0KICAgICgyNTYsIDEyOCwgNjQpOiA1NV82ODcsICAgICMgTS1kZWVwICAodHXhu7MgY2jhu41uKQ0KfQ0KDQoNCmNsYXNzIE1MUChubi5Nb2R1bGUpOg0KICAgICIiIk1MUCB0aGVvIHF1eSDEkeG7i25oIOG7nyDEkeG6p3UgZmlsZS4NCg0KICAgIEFyZ3M6DQogICAgICAgIGhpZGRlbjogICB0dXBsZSBz4buRIG7GoS1yb24gY8OhYyBs4bubcCDhuqluLCB2w60gZOG7pSAoMjU2LCAxMjgpDQogICAgICAgIGRyb3BvdXQ6ICB4w6FjIHN14bqldCBU4bquVCBuxqEtcm9uIHEgKG5uLkRyb3BvdXQgZMO5bmcgcCBjaMOtbmggbMOgIHjDoWMgc3XhuqV0IHThuq90KTsgMC4wID0ga2jDtG5nIGTDuW5nDQogICAgICAgIGluaXQ6ICAgICAiemVyb3MiIHwgIm5vcm1hbCIgfCAieGF2aWVyIiB8ICJoZSIgfCAiZGVmYXVsdCINCiAgICAiIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBoaWRkZW49KDI1NiwgMTI4KSwgZHJvcG91dDogZmxvYXQgPSAwLjAsIGluaXQ6IHN0ciA9ICJoZSIsDQogICAgICAgICAgICAgICAgIGluX2ZlYXR1cmVzOiBpbnQgPSA1NCwgbnVtX2NsYXNzZXM6IGludCA9IDcpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgaGlkZGVuID0gdHVwbGUoaGlkZGVuKQ0KICAgICAgICBpZiBub3QgaGlkZGVuIG9yIGFueShpbnQod2lkdGgpIDw9IDAgZm9yIHdpZHRoIGluIGhpZGRlbik6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJoaWRkZW4gbXVzdCBjb250YWluIHBvc2l0aXZlIGxheWVyIHdpZHRocyIpDQogICAgICAgIGlmIG5vdCAwLjAgPD0gZHJvcG91dCA8IDEuMDoNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImRyb3BvdXQgbXVzdCBiZSBpbiBbMCwgMSkiKQ0KICAgICAgICBpZiBpbl9mZWF0dXJlcyA8PSAwIG9yIG51bV9jbGFzc2VzIDw9IDA6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbl9mZWF0dXJlcyBhbmQgbnVtX2NsYXNzZXMgbXVzdCBiZSBwb3NpdGl2ZSIpDQoNCiAgICAgICAgbGF5ZXJzOiBsaXN0W25uLk1vZHVsZV0gPSBbXQ0KICAgICAgICB3aWR0aF9pbiA9IGluX2ZlYXR1cmVzDQogICAgICAgIGZvciB3aWR0aF9vdXQgaW4gaGlkZGVuOg0KICAgICAgICAgICAgbGF5ZXJzLmV4dGVuZCgobm4uTGluZWFyKHdpZHRoX2luLCB3aWR0aF9vdXQpLCBubi5SZUxVKCkpKQ0KICAgICAgICAgICAgaWYgZHJvcG91dCA+IDAuMDoNCiAgICAgICAgICAgICAgICBsYXllcnMuYXBwZW5kKG5uLkRyb3BvdXQocD1kcm9wb3V0KSkNCiAgICAgICAgICAgIHdpZHRoX2luID0gd2lkdGhfb3V0DQogICAgICAgIGxheWVycy5hcHBlbmQobm4uTGluZWFyKHdpZHRoX2luLCBudW1fY2xhc3NlcykpDQogICAgICAgIHNlbGYubmV0ID0gbm4uU2VxdWVudGlhbCgqbGF5ZXJzKQ0KICAgICAgICBpbml0X3dlaWdodHMoc2VsZiwgaW5pdCkNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOg0KICAgICAgICAiIiJ4OiAoQiwgNTQpIGZsb2F0MzIgIC0+ICBsb2dpdHM6IChCLCA3KSBmbG9hdDMyLiIiIg0KICAgICAgICByZXR1cm4gc2VsZi5uZXQoeCkNCg0KDQpkZWYgaW5pdF93ZWlnaHRzKG1vZGVsOiBubi5Nb2R1bGUsIGluaXQ6IHN0cikgLT4gTm9uZToNCiAgICAiIiJLaOG7n2kgdOG6oW8gdGhhbSBz4buRIGPhu6dhIE3hu4xJIG5uLkxpbmVhciAoYmlhcyA9IDAgduG7m2kgaW5pdCB0xrDhu51uZyBtaW5oKS4NCg0KICAgIGluaXQ6DQogICAgICAgICJ6ZXJvcyIgICA6IFcgPSAwDQogICAgICAgICJub3JtYWwiICA6IFcgfiBOKDAsIDAuMDFeMikNCiAgICAgICAgInhhdmllciIgIDogbm4uaW5pdC54YXZpZXJfbm9ybWFsXyAoVmFyID0gMi8obl9pbituX291dCkpOyBu4bq/dSBi4bqhbiBkw7luZyBWYXIgPSAxL25faW4gdGhlbyBzbGlkZSwgaMOjeSBnaGkgcsO1DQogICAgICAgICJoZSIgICAgICA6IG5uLmluaXQua2FpbWluZ19ub3JtYWxfKHcsIG5vbmxpbmVhcml0eT0icmVsdSIpICAoVmFyID0gMi9uX2luKQ0KICAgICAgICAiZGVmYXVsdCIgOiBraMO0bmcgbMOgbSBnw6wgKGdp4buvIGto4bufaSB04bqhbyBt4bq3YyDEkeG7i25oIGPhu6dhIG5uLkxpbmVhcjsgS0jDlE5HIHBo4bqjaSBIZSkNCiAgICBH4bujaSDDvTogZHV54buHdCBtb2RlbC5tb2R1bGVzKCksIGNo4buNbiBpc2luc3RhbmNlKG0sIG5uLkxpbmVhcikuDQogICAgIiIiDQogICAgdmFsaWQgPSB7Inplcm9zIiwgIm5vcm1hbCIsICJ4YXZpZXIiLCAiaGUiLCAiZGVmYXVsdCJ9DQogICAgaWYgaW5pdCBub3QgaW4gdmFsaWQ6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJpbml0IG11c3QgYmUgb25lIG9mIHtzb3J0ZWQodmFsaWQpfSwgZ290IHtpbml0IXJ9IikNCiAgICBpZiBpbml0ID09ICJkZWZhdWx0IjoNCiAgICAgICAgcmV0dXJuDQoNCiAgICBmb3IgbW9kdWxlIGluIG1vZGVsLm1vZHVsZXMoKToNCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UobW9kdWxlLCBubi5MaW5lYXIpOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgaWYgaW5pdCA9PSAiemVyb3MiOg0KICAgICAgICAgICAgbm4uaW5pdC56ZXJvc18obW9kdWxlLndlaWdodCkNCiAgICAgICAgZWxpZiBpbml0ID09ICJub3JtYWwiOg0KICAgICAgICAgICAgbm4uaW5pdC5ub3JtYWxfKG1vZHVsZS53ZWlnaHQsIG1lYW49MC4wLCBzdGQ9MC4wMSkNCiAgICAgICAgZWxpZiBpbml0ID09ICJ4YXZpZXIiOg0KICAgICAgICAgICAgbm4uaW5pdC54YXZpZXJfbm9ybWFsXyhtb2R1bGUud2VpZ2h0KQ0KICAgICAgICBlbGlmIGluaXQgPT0gImhlIjoNCiAgICAgICAgICAgIG5uLmluaXQua2FpbWluZ19ub3JtYWxfKG1vZHVsZS53ZWlnaHQsIG5vbmxpbmVhcml0eT0icmVsdSIpDQogICAgICAgIGlmIG1vZHVsZS5iaWFzIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgbm4uaW5pdC56ZXJvc18obW9kdWxlLmJpYXMpDQoNCg0KZGVmIGNvdW50X3BhcmFtcyhtb2RlbDogbm4uTW9kdWxlKSAtPiBpbnQ6DQogICAgIiIiVOG7lW5nIHPhu5EgdGhhbSBz4buRIGh14bqlbiBsdXnhu4duIMSRxrDhu6NjLiBEw7luZyDEkeG7gyBhc3NlcnQgduG7m2kgRVhQRUNURURfUEFSQU1TIG5nYXkgc2F1IGtoaSB04bqhbyBtb2RlbC4iIiINCiAgICByZXR1cm4gc3VtKHBhcmFtZXRlci5udW1lbCgpIGZvciBwYXJhbWV0ZXIgaW4gbW9kZWwucGFyYW1ldGVycygpKQ0KDQoNCkB0b3JjaC5ub19ncmFkKCkNCmRlZiBhY3RpdmF0aW9uX3N0YXRzKG1vZGVsOiBubi5Nb2R1bGUsIHg6IHRvcmNoLlRlbnNvcikgLT4gbGlzdFtmbG9hdF06DQogICAgIiIixJDhu5kgbOG7h2NoIGNodeG6qW4gY+G7p2Ega8OtY2ggaG/huqF0IHNhdSBt4buXaSBs4bubcCAo4bufIGLGsOG7m2MgMCwgbeG7mXQgbMO0IHZhbCkg4oCUIGTDuW5nIGNobyB0aMOtIG5naGnhu4dtIGto4bufaSB04bqhby4NCg0KICAgIEPDoWMgYsaw4bubYzoNCiAgICAgIDEuIG1vZGVsLmV2YWwoKTsgaCA9IHgNCiAgICAgIDIuIGR1eeG7h3QgdOG7q25nIGzhu5twIGNvbiB0aGVvIHRo4bupIHThu7E7IHNhdSBt4buXaSBubi5MaW5lYXIgKGhv4bq3YyBzYXUgbeG7l2kgUmVMVSwgYuG6oW4gY2jhu41uIHbDoCBnaGkgcsO1KSBsxrB1IGguc3RkKCkuaXRlbSgpDQogICAgICAzLiB0cuG6oyB24buBIGRhbmggc8OhY2ggc3RkIHRoZW8gbOG7m3ANCiAgICAiIiINCiAgICB3YXNfdHJhaW5pbmcgPSBtb2RlbC50cmFpbmluZw0KICAgIG1vZGVsLmV2YWwoKQ0KICAgIGFjdGl2YXRpb25zOiBsaXN0W2Zsb2F0XSA9IFtdDQogICAgaCA9IHgNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgZm9yIGxheWVyIGluIG1vZGVsLm5ldDoNCiAgICAgICAgICAgIGggPSBsYXllcihoKQ0KICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShsYXllciwgbm4uUmVMVSk6DQogICAgICAgICAgICAgICAgYWN0aXZhdGlvbnMuYXBwZW5kKGZsb2F0KGguc3RkKHVuYmlhc2VkPUZhbHNlKS5pdGVtKCkpKQ0KICAgIG1vZGVsLnRyYWluKHdhc190cmFpbmluZykNCiAgICByZXR1cm4gYWN0aXZhdGlvbnMNCg==","optimizer.py":"IiIiT3B0aW1pemVyIGNvbnN0cnVjdGlvbiBhbmQgZ2xvYmFsIGdyYWRpZW50IGNsaXBwaW5nIGhlbHBlcnMuDQoNCsSQxrDhu6NjIGTDuW5nIHRvcmNoLm9wdGltLiogdsOgIHRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyAoeGVtIFJFQURNRSBt4bulYyA1KS4NCkZpbGUgbsOgeSBnb20gdmnhu4djIGNo4buNbiBi4buZIHThu5FpIMawdSB2w6AgY+G6r3QgZ3JhZGllbnQgxJHhu4MgYHRyYWluLnB5YCBn4buNbiB2w6AgbeG7jWkgdGjDrSBuZ2hp4buHbSBjw7RuZyBi4bqxbmcuDQoNCkPDtG5nIHRo4bupYyBj4bqnbiBoaeG7g3UgKHNsaWRlIENoxrDGoW5nIDQpOg0KICAgIFNHRCAgICAgICAgICAgIDogdyA8LSB3IC0gbHIgKiBnDQogICAgU0dEICsgbW9tZW50dW0gOiB2IDwtIG11ICogdiArIGcgOyAgdyA8LSB3IC0gbHIgKiB2ICAgICAgICAgIChk4bqhbmcgUHlUb3JjaCkNCiAgICBBZGFtICAgICAgICAgICA6IG0gPC0gYjEgbSArICgxLWIxKSBnIDsgdiA8LSBiMiB2ICsgKDEtYjIpIGdeMiA7IHcgPC0gdyAtIGxyICogbV9oYXQgLyAoc3FydCh2X2hhdCkgKyBlcHMpDQogICAgQWRhbVcgICAgICAgICAgOiBuaMawIEFkYW0gbmjGsG5nIHN1eSBnaeG6o20gdHLhu41uZyBz4buRIHTDoWNoIHJpw6puZzogdyA8LSB3IC0gbHIgKiB3ZCAqIHcgLSBsciAqIG1faGF0IC8gKHNxcnQodl9oYXQpICsgZXBzKQ0KIiIiDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCB0b3JjaA0KDQpPUFRJTUlaRVJTID0gKCJzZ2QiLCAic2dkX21vbWVudHVtIiwgImFkYW0iLCAiYWRhbXciKQ0KDQoNCmRlZiBidWlsZF9vcHRpbWl6ZXIobmFtZTogc3RyLCBwYXJhbXMsIGxyOiBmbG9hdCwgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDAuMCwNCiAgICAgICAgICAgICAgICAgICAgbW9tZW50dW06IGZsb2F0ID0gMC45LCBiZXRhcz0oMC45LCAwLjk5OSksIGVwczogZmxvYXQgPSAxZS04KToNCiAgICAiIiJUcuG6oyB24buBIG3hu5l0IHRvcmNoLm9wdGltLk9wdGltaXplci4NCg0KICAgIEPDoWMgYsaw4bubYzoNCiAgICAgIDEuIGtp4buDbSB0cmEgbmFtZSBu4bqxbSB0cm9uZyBPUFRJTUlaRVJTLCBu4bq/dSBraMO0bmcgcmFpc2UgVmFsdWVFcnJvcg0KICAgICAgMi4gInNnZCIgICAgICAgICAgLT4gdG9yY2gub3B0aW0uU0dEKHBhcmFtcywgbHI9bHIsIHdlaWdodF9kZWNheT13ZWlnaHRfZGVjYXkpDQogICAgICAgICAic2dkX21vbWVudHVtIiAtPiB0b3JjaC5vcHRpbS5TR0QocGFyYW1zLCBscj1sciwgbW9tZW50dW09bW9tZW50dW0sIHdlaWdodF9kZWNheT13ZWlnaHRfZGVjYXkpDQogICAgICAgICAiYWRhbSIgICAgICAgICAtPiB0b3JjaC5vcHRpbS5BZGFtKHBhcmFtcywgbHI9bHIsIGJldGFzPWJldGFzLCBlcHM9ZXBzLCB3ZWlnaHRfZGVjYXk9d2VpZ2h0X2RlY2F5KQ0KICAgICAgICAgImFkYW13IiAgICAgICAgLT4gdG9yY2gub3B0aW0uQWRhbVcocGFyYW1zLCBscj1sciwgYmV0YXM9YmV0YXMsIGVwcz1lcHMsIHdlaWdodF9kZWNheT13ZWlnaHRfZGVjYXkpDQogICAgQ2jDuiDDvTogd2VpZ2h0X2RlY2F5IGPhu6dhIEFkYW0gKEwyIHRy4buZbiB2w6BvIGdyYWRpZW50KSBraMOhYyB3ZWlnaHRfZGVjYXkgY+G7p2EgQWRhbVcgKHN1eSBnaeG6o20gdMOhY2ggcmnDqm5nKS4NCiAgICAiIiINCiAgICBpZiBuYW1lIG5vdCBpbiBPUFRJTUlaRVJTOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5rbm93biBvcHRpbWl6ZXIge25hbWUhcn07IGNob29zZSBvbmUgb2Yge09QVElNSVpFUlN9IikNCiAgICBpZiBuYW1lID09ICJzZ2QiOg0KICAgICAgICByZXR1cm4gdG9yY2gub3B0aW0uU0dEKHBhcmFtcywgbHI9bHIsIHdlaWdodF9kZWNheT13ZWlnaHRfZGVjYXkpDQogICAgaWYgbmFtZSA9PSAic2dkX21vbWVudHVtIjoNCiAgICAgICAgcmV0dXJuIHRvcmNoLm9wdGltLlNHRChwYXJhbXMsIGxyPWxyLCBtb21lbnR1bT1tb21lbnR1bSwgd2VpZ2h0X2RlY2F5PXdlaWdodF9kZWNheSkNCiAgICBpZiBuYW1lID09ICJhZGFtIjoNCiAgICAgICAgcmV0dXJuIHRvcmNoLm9wdGltLkFkYW0ocGFyYW1zLCBscj1sciwgYmV0YXM9YmV0YXMsIGVwcz1lcHMsIHdlaWdodF9kZWNheT13ZWlnaHRfZGVjYXkpDQogICAgcmV0dXJuIHRvcmNoLm9wdGltLkFkYW1XKHBhcmFtcywgbHI9bHIsIGJldGFzPWJldGFzLCBlcHM9ZXBzLCB3ZWlnaHRfZGVjYXk9d2VpZ2h0X2RlY2F5KQ0KDQoNCmRlZiBidWlsZF9zY2hlZHVsZXIob3B0aW1pemVyLCBuYW1lOiBzdHIgfCBOb25lLCB0b3RhbF9zdGVwczogaW50LCAqKmt3YXJncyk6DQogICAgIiIiKFR14buzIGNo4buNbikgQuG7mSBs4bqtcCBs4buLY2ggdOG7kWMgxJHhu5kgaOG7jWMsIHbDrSBk4bulIGNvc2luZSAoc2xpZGUgY8OzIHbDrSBk4bulIENvc2luZUFubmVhbGluZ0xSKS4NCg0KICAgIFRy4bqjIHbhu4EgTm9uZSBu4bq/dSBuYW1lIGzDoCBOb25lLiBO4bq/dSBi4bqhbiBkw7luZyBzY2hlZHVsZXIg4bufIG3hu5l0IHRow60gbmdoaeG7h20sIGjDo3kgZ2hpIHbDoG8gYuG6o25nIChj4buZdCBub3RlcykuDQogICAgIiIiDQogICAgaWYgbmFtZSBpcyBOb25lOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIGlmIHRvdGFsX3N0ZXBzIDw9IDA6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRvdGFsX3N0ZXBzIG11c3QgYmUgcG9zaXRpdmUiKQ0KICAgIGlmIG5hbWUgPT0gImNvc2luZSI6DQogICAgICAgIHJldHVybiB0b3JjaC5vcHRpbS5scl9zY2hlZHVsZXIuQ29zaW5lQW5uZWFsaW5nTFIoDQogICAgICAgICAgICBvcHRpbWl6ZXIsIFRfbWF4PXRvdGFsX3N0ZXBzLCAqKmt3YXJncw0KICAgICAgICApDQogICAgcmFpc2UgVmFsdWVFcnJvcigiU3VwcG9ydGVkIHNjaGVkdWxlcnMgYXJlIE5vbmUgYW5kICdjb3NpbmUnIikNCg0KDQpkZWYgY2xpcF9ncmFkaWVudHMocGFyYW1zLCBtYXhfbm9ybTogZmxvYXQgfCBOb25lKSAtPiBmbG9hdDoNCiAgICAiIiJD4bqvdCBncmFkaWVudCB0aGVvIGNodeG6qW4gTDIgdG/DoG4gY+G7pWMsIHbDoCBUUuG6oiBW4buAIGNodeG6qW4gZ3JhZGllbnQgVFLGr+G7mkMgS0hJIGPhuq90Lg0KDQogICAgQ8OhYyBixrDhu5tjOg0KICAgICAgMS4gbuG6v3UgbWF4X25vcm0gbMOgIE5vbmU6IHTDrW5oIGNodeG6qW4gdG/DoG4gY+G7pWMgbcOgIGtow7RuZyBj4bqvdCAodsOtIGThu6UgY2xpcF9ncmFkX25vcm1fIHbhu5tpIG1heF9ub3JtPWluZikNCiAgICAgIDIuIG5nxrDhu6NjIGzhuqFpOiB0b3RhbF9ub3JtID0gdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKHBhcmFtcywgbWF4X25vcm0pDQogICAgICAzLiByZXR1cm4gZmxvYXQodG90YWxfbm9ybSkNCiAgICBHacOhIHRy4buLIHRy4bqjIHbhu4EgY2jDrW5oIGzDoCBgZ3JhZF9ub3JtYCBi4bqhbiBwaOG6o2kgZ2hpIGzhuqFpIOG7nyBt4buXaSBixrDhu5tjICjEkeG7gyB0aOG6pXkgImdhaSIgZ3JhZGllbnQpLg0KICAgIEtoaSBkw7luZyBtaXhlZCBwcmVjaXNpb24gRlAxNiArIEdyYWRTY2FsZXI6IHBo4bqjaSBzY2FsZXIudW5zY2FsZV8ob3B0aW1pemVyKSBUUsav4buaQyBraGkgZ+G7jWkgaMOgbSBuw6B5Lg0KICAgICIiIg0KICAgIGlmIG1heF9ub3JtIGlzIG5vdCBOb25lIGFuZCBtYXhfbm9ybSA8PSAwOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtYXhfbm9ybSBtdXN0IGJlIHBvc2l0aXZlIG9yIE5vbmUiKQ0KICAgIG5vcm1fbGltaXQgPSBmbG9hdCgiaW5mIikgaWYgbWF4X25vcm0gaXMgTm9uZSBlbHNlIGZsb2F0KG1heF9ub3JtKQ0KICAgIHRvdGFsX25vcm0gPSB0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8ocGFyYW1zLCBub3JtX2xpbWl0KQ0KICAgIHJldHVybiBmbG9hdCh0b3RhbF9ub3JtLml0ZW0oKSBpZiBpc2luc3RhbmNlKHRvdGFsX25vcm0sIHRvcmNoLlRlbnNvcikgZWxzZSB0b3RhbF9ub3JtKQ0K","train.py":"...","plots.py":"IiIiUGxvdHRpbmcgaGVscGVycyBmb3IgaW5kaXZpZHVhbCBydW5zIGFuZCBleHBlcmltZW50IGNvbXBhcmlzb25zLg0KDQrhuqJuaCBiaeG7g3UgxJHhu5MgbMOgIHPhuqNuIHBo4bqpbSBu4buZcCAoeGVtIFJFQURNRSBt4bulYyA2KTogbeG7l2kgdGjDrSBuZ2hp4buHbSBt4buZdCDhuqNuaCBmaWd1cmVzLzxleHBfaWQ+LnBuZy4NCktoaSBub3RlYm9vayBjaOG6oXkgdHJvbmcgY29kZS8sIGzGsHUgdsOgbyAiLi4vZmlndXJlcy8iICh2w60gZOG7pSBwYXRoID0gZiIuLi9maWd1cmVzL3tleHBfaWR9LnBuZyIpLg0KIiIiDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KDQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0DQoNCg0KZGVmIHBsb3RfcnVuKHJlc3VsdDogZGljdCwgcGF0aDogc3RyKSAtPiBOb25lOg0KICAgICIiIlbhur0gTeG7mFQgdGjDrSBuZ2hp4buHbSB0aMOgbmggbeG7mXQg4bqjbmggUE5HIGPDsyDDrXQgbmjhuqV0IDMgw7Q6DQogICAgICAgICAoMSkgdHJhaW5fbG9zcyB2w6AgdmFsX2xvc3MgdGhlbyBlcG9jaCAoY8O5bmcgbeG7mXQgdHLhu6VjKQ0KICAgICAgICAgKDIpIHZhbF9hY2MgKHbDoCBuw6puIGPDsyB2YWxfbWFjcm9fZjEpIHRoZW8gZXBvY2gNCiAgICAgICAgICgzKSBncmFkX25vcm0gdGhlbyBlcG9jaCAoxJFvIFRSxq/hu5pDIGtoaSBjbGlwKQ0KICAgIFnDqnUgY+G6p3U6IHRpw6p1IMSR4buBIGdoaSBleHBfaWQgdsOgIGPhuqV1IGjDrG5oIGNow61uaCAob3B0aW1pemVyLCBsciwgYmF0Y2gsIC4uLiksIGPDsyBuaMOjbiB0cuG7pWMgdsOgIGNow7ogdGjDrWNoLg0KICAgIEPDoWMgYsaw4bubYzogZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDMsIGZpZ3NpemU9Li4uKTsgcGxvdDsgc2V0X3RpdGxlL3hsYWJlbC9sZWdlbmQ7DQogICAgICAgICAgICAgIGZpZy5zYXZlZmlnKHBhdGgsIGRwaT0uLi4sIGJib3hfaW5jaGVzPSJ0aWdodCIpOyBwbHQuY2xvc2UoZmlnKQ0KICAgIEfhu6NpIMO9OiDEkcOhbmggZOG6pXUgYmVzdF9lcG9jaCBi4bqxbmcgxJHGsOG7nW5nIHRo4bqzbmcgxJHhu6luZy4NCiAgICAiIiINCiAgICBjZmcgPSByZXN1bHRbImNmZyJdDQogICAgaGlzdG9yeSA9IHJlc3VsdFsiaGlzdG9yeSJdDQogICAgZXBvY2hzID0gaGlzdG9yeVsiZXBvY2giXQ0KICAgIGV4cF9pZCA9IGNmZy5nZXQoImV4cF9pZCIsICJleHBlcmltZW50IikNCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMSwgMywgZmlnc2l6ZT0oMTYsIDQuNSkpDQoNCiAgICBpZiBub3QgZXBvY2hzOg0KICAgICAgICBmb3IgYXhpcywgbGFiZWwgaW4gemlwKGF4ZXMsICgiTG9zcyIsICJWYWxpZGF0aW9uIG1ldHJpY3MiLCAiR3JhZGllbnQgbm9ybSIpKToNCiAgICAgICAgICAgIGF4aXMuc2V0X3RpdGxlKGxhYmVsKQ0KICAgICAgICAgICAgYXhpcy5zZXRfeGxhYmVsKCJFcG9jaCIpDQogICAgICAgICAgICBheGlzLnRleHQoMC41LCAwLjUsICJObyBjb21wbGV0ZWQgZXBvY2hzXG5SdW4gZGl2ZXJnZWQgZHVyaW5nIHRyYWluaW5nIiwNCiAgICAgICAgICAgICAgICAgICAgICBoYT0iY2VudGVyIiwgdmE9ImNlbnRlciIsIHRyYW5zZm9ybT1heGlzLnRyYW5zQXhlcykNCiAgICAgICAgZmlnLnN1cHRpdGxlKGYie2V4cF9pZH06IHRyYWluaW5nIGRpdmVyZ2VkIGJlZm9yZSBmaXJzdCBlcG9jaCBjb21wbGV0ZWQiKQ0KICAgICAgICBmaWcudGlnaHRfbGF5b3V0KCkNCiAgICAgICAgb3V0cHV0ID0gUGF0aChwYXRoKQ0KICAgICAgICBvdXRwdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICAgICAgZmlnLnNhdmVmaWcob3V0cHV0LCBkcGk9MTYwLCBiYm94X2luY2hlcz0idGlnaHQiKQ0KICAgICAgICBwbHQuY2xvc2UoZmlnKQ0KICAgICAgICByZXR1cm4NCg0KICAgIGF4ZXNbMF0ucGxvdChlcG9jaHMsIGhpc3RvcnlbInRyYWluX2xvc3MiXSwgbWFya2VyPSJvIiwgbGFiZWw9InRyYWluIikNCiAgICBheGVzWzBdLnBsb3QoZXBvY2hzLCBoaXN0b3J5WyJ2YWxfbG9zcyJdLCBtYXJrZXI9Im8iLCBsYWJlbD0idmFsaWRhdGlvbiIpDQogICAgYXhlc1swXS5zZXRfdGl0bGUoIkxvc3MgKGV2YWwgbW9kZSkiKQ0KICAgIGF4ZXNbMF0uc2V0X3hsYWJlbCgiRXBvY2giKQ0KICAgIGF4ZXNbMF0uc2V0X3lsYWJlbCgiTG9zcyIpDQogICAgYXhlc1swXS5sZWdlbmQoKQ0KDQogICAgYXhlc1sxXS5wbG90KGVwb2NocywgaGlzdG9yeVsidmFsX2FjYyJdLCBtYXJrZXI9Im8iLCBsYWJlbD0iYWNjdXJhY3kiKQ0KICAgIGlmICJ2YWxfbWFjcm9fZjEiIGluIGhpc3Rvcnk6DQogICAgICAgIGF4ZXNbMV0ucGxvdChlcG9jaHMsIGhpc3RvcnlbInZhbF9tYWNyb19mMSJdLCBtYXJrZXI9Im8iLCBsYWJlbD0ibWFjcm8tRjEiKQ0KICAgIGF4ZXNbMV0uc2V0X3RpdGxlKCJWYWxpZGF0aW9uIG1ldHJpY3MiKQ0KICAgIGF4ZXNbMV0uc2V0X3hsYWJlbCgiRXBvY2giKQ0KICAgIGF4ZXNbMV0uc2V0X3lsYWJlbCgiU2NvcmUiKQ0KICAgIGF4ZXNbMV0uc2V0X3lsaW0oMCwgMSkNCiAgICBheGVzWzFdLmxlZ2VuZCgpDQoNCiAgICBheGVzWzJdLnBsb3QoZXBvY2hzLCBoaXN0b3J5WyJncmFkX25vcm0iXSwgbWFya2VyPSJvIiwgbGFiZWw9InByZS1jbGlwIG5vcm0iKQ0KICAgIGF4ZXNbMl0uc2V0X3RpdGxlKCJHcmFkaWVudCBub3JtIikNCiAgICBheGVzWzJdLnNldF94bGFiZWwoIkVwb2NoIikNCiAgICBheGVzWzJdLnNldF95bGFiZWwoIkdsb2JhbCBMMiBub3JtIikNCiAgICBheGVzWzJdLmxlZ2VuZCgpDQoNCiAgICBiZXN0X2Vwb2NoID0gcmVzdWx0WyJzdW1tYXJ5Il0uZ2V0KCJiZXN0X2Vwb2NoIikNCiAgICBpZiBiZXN0X2Vwb2NoIGluIGVwb2NoczoNCiAgICAgICAgZm9yIGF4aXMgaW4gYXhlczoNCiAgICAgICAgICAgIGF4aXMuYXh2bGluZShiZXN0X2Vwb2NoLCBjb2xvcj0iZ3JheSIsIGxpbmVzdHlsZT0iLS0iLCBhbHBoYT0wLjYpDQogICAgY29uZmlnX3RleHQgPSAoDQogICAgICAgIGYie2NmZy5nZXQoJ29wdGltaXplcicpfSDCtyBscj17Y2ZnLmdldCgnbHInKX0gwrcgYmF0Y2g9e2NmZy5nZXQoJ2JhdGNoJyl9IMK3ICINCiAgICAgICAgZiJsb3NzPXtjZmcuZ2V0KCdsb3NzJyl9IMK3IGluaXQ9e2NmZy5nZXQoJ2luaXQnKX0iDQogICAgKQ0KICAgIGZpZy5zdXB0aXRsZShmIntleHBfaWR9OiB7Y29uZmlnX3RleHR9IikNCiAgICBmaWcudGlnaHRfbGF5b3V0KCkNCiAgICBvdXRwdXQgPSBQYXRoKHBhdGgpDQogICAgb3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgZmlnLnNhdmVmaWcob3V0cHV0LCBkcGk9MTYwLCBiYm94X2luY2hlcz0idGlnaHQiKQ0KICAgIHBsdC5jbG9zZShmaWcpDQoNCg0KZGVmIHBsb3RfY29tcGFyZShyZXN1bHRzOiBsaXN0W2RpY3RdLCBtZXRyaWM6IHN0ciwgcGF0aDogc3RyLCB0aXRsZTogc3RyID0gIiIpIC0+IE5vbmU6DQogICAgIiIiVuG6vSBjaOG7k25nIG3hu5l0IGNo4buJIHPhu5EgKHbDrSBk4bulICJ2YWxfbG9zcyIsICJ2YWxfbWFjcm9fZjEiLCAiZ3JhZF9ub3JtIikgY+G7p2Egbmhp4buBdSB0aMOtIG5naGnhu4dtDQogICAgdHLDqm4gY8O5bmcgbeG7mXQgdHLhu6VjLCBt4buXaSB0aMOtIG5naGnhu4dtIG3hu5l0IMSRxrDhu51uZywgY2jDuiB0aMOtY2ggYuG6sW5nIGV4cF9pZC4NCg0KICAgIETDuW5nIGNobyDhuqNuaCBmaWd1cmVzL2NvbXBhcmVfPG5ow7NtPi5wbmcgKHbDrSBk4bulIGNvbXBhcmVfb3B0aW1pemVyLnBuZykuDQogICAgIiIiDQogICAgaWYgbm90IHJlc3VsdHM6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlc3VsdHMgbXVzdCBjb250YWluIGF0IGxlYXN0IG9uZSBydW4iKQ0KICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oOCwgNSkpDQogICAgZm9yIHJlc3VsdCBpbiByZXN1bHRzOg0KICAgICAgICBoaXN0b3J5ID0gcmVzdWx0WyJoaXN0b3J5Il0NCiAgICAgICAgaWYgbWV0cmljIG5vdCBpbiBoaXN0b3J5Og0KICAgICAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJ7bWV0cmljIXJ9IGlzIG5vdCBpbiB0aGUgcnVuIGhpc3RvcnkiKQ0KICAgICAgICBheC5wbG90KGhpc3RvcnlbImVwb2NoIl0sIGhpc3RvcnlbbWV0cmljXSwgbWFya2VyPSJvIiwgbGFiZWw9cmVzdWx0WyJjZmciXS5nZXQoImV4cF9pZCIsICJydW4iKSkNCiAgICBheC5zZXRfdGl0bGUodGl0bGUgb3IgbWV0cmljLnJlcGxhY2UoIl8iLCAiICIpLnRpdGxlKCkpDQogICAgYXguc2V0X3hsYWJlbCgiRXBvY2giKQ0KICAgIGF4LnNldF95bGFiZWwobWV0cmljLnJlcGxhY2UoIl8iLCAiICIpKQ0KICAgIGF4LmxlZ2VuZCgpDQogICAgYXguZ3JpZChhbHBoYT0wLjI1KQ0KICAgIG91dHB1dCA9IFBhdGgocGF0aCkNCiAgICBvdXRwdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBmaWcudGlnaHRfbGF5b3V0KCkNCiAgICBmaWcuc2F2ZWZpZyhvdXRwdXQsIGRwaT0xNjAsIGJib3hfaW5jaGVzPSJ0aWdodCIpDQogICAgcGx0LmNsb3NlKGZpZykNCg==","results_table.py":"IiIiUGVyc2lzdCBleHBlcmltZW50IGhpc3RvcmllcyBhbmQgcHJlcGFyZSB0aGUgc3VibWlzc2lvbiB0YWJsZS4NCg0KTmhp4buHbSB24bulOiBsxrB1IGvhur90IHF14bqjIHThu6tuZyBs4bqnbiBjaOG6oXkgcmEgSlNPTiwgcuG7k2kgxJFp4buBbiB2w6BvIGV4cGVyaW1lbnRzLnhsc3ggdOG7qyBt4bqrdQ0KdGVtcGxhdGVzL2V4cGVyaW1lbnRfdGFibGVfdGVtcGxhdGUueGxzeCAoxJHhu6tuZyBnw7UgdGF5IGjDoG5nIGNo4bulYyBkw7JuZywgcuG6pXQgZOG7hSBzYWkpLg0KDQpUw6puIGPhu5l0IGPhu6dhIHNoZWV0ICJFeHBlcmltZW50cyIgKGdp4buvIG5ndXnDqm4sIMSRw7puZyB0aOG7qSB04buxIG3huqt1KToNCiAgICBleHBfaWQsIGdyb3VwLCBkZXNjcmlwdGlvbiwgbG9zcywgb3B0aW1pemVyLCBsciwgd2VpZ2h0X2RlY2F5LCBiYXRjaCwgZXBvY2hzLCBoaWRkZW4sIGRyb3BvdXQsDQogICAgY2xpcF9ub3JtLCBwcmVjaXNpb24sIGluaXQsIHNlZWQsIHN0ZXAwX2xvc3MsIGJlc3RfdmFsX2xvc3MsIGJlc3RfZXBvY2gsIGZpbmFsX3RyYWluX2xvc3MsDQogICAgZmluYWxfdmFsX2xvc3MsIHZhbF9hY2MsIHZhbF9tYWNyb19mMSwgdGltZV9wZXJfZXBvY2hfcywgcGVha19tZW1fTUIsIGRpdmVyZ2VkLA0KICAgIGV2YWxfYWNjLCBldmFsX21hY3JvX2YxLCBmaWd1cmVfZmlsZSwgbm90ZXMNCihjw6FjIGPhu5l0IGPDtG5nIHRo4bupYyDhu58gY3Xhu5FpIGLhuqNuZyBt4bqrdSB04buxIHTDrW5oLCDEkeG7q25nIGdoaSDEkcOoKQ0KIiIiDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBqc29uDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KDQpkZWYgc2F2ZV9yZXN1bHQocmVzdWx0OiBkaWN0LCByZXN1bHRzX2Rpcjogc3RyID0gIi4uL3Jlc3VsdHMiKSAtPiBzdHI6DQogICAgIiIiR2hpIHJlc3VsdFsiY2ZnIl0sIHJlc3VsdFsiaGlzdG9yeSJdLCByZXN1bHRbInN1bW1hcnkiXSAoS0jDlE5HIGdoaSBiZXN0X3N0YXRlKSByYQ0KICAgIDxyZXN1bHRzX2Rpcj4vPGV4cF9pZD4uanNvbi4gVHLhuqMgduG7gSDEkcaw4budbmcgZOG6q24gZmlsZS4gVOG6oW8gdGjGsCBt4bulYyBu4bq/dSBjaMawYSBjw7MuIiIiDQogICAgb3V0cHV0X2RpciA9IFBhdGgocmVzdWx0c19kaXIpDQogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgZXhwX2lkID0gc3RyKHJlc3VsdFsiY2ZnIl1bImV4cF9pZCJdKQ0KICAgIHBheWxvYWQgPSB7DQogICAgICAgICJjZmciOiByZXN1bHRbImNmZyJdLA0KICAgICAgICAiaGlzdG9yeSI6IHJlc3VsdFsiaGlzdG9yeSJdLA0KICAgICAgICAic3VtbWFyeSI6IHJlc3VsdFsic3VtbWFyeSJdLA0KICAgIH0NCg0KICAgIGRlZiBqc29uX2RlZmF1bHQodmFsdWUpOg0KICAgICAgICBpZiBoYXNhdHRyKHZhbHVlLCAidG9saXN0Iik6DQogICAgICAgICAgICByZXR1cm4gdmFsdWUudG9saXN0KCkNCiAgICAgICAgaWYgaGFzYXR0cih2YWx1ZSwgIml0ZW0iKToNCiAgICAgICAgICAgIHJldHVybiB2YWx1ZS5pdGVtKCkNCiAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgUGF0aCk6DQogICAgICAgICAgICByZXR1cm4gc3RyKHZhbHVlKQ0KICAgICAgICByYWlzZSBUeXBlRXJyb3IoZiJPYmplY3Qgb2YgdHlwZSB7dHlwZSh2YWx1ZSkuX19uYW1lX199IGlzIG5vdCBKU09OIHNlcmlhbGl6YWJsZSIpDQoNCiAgICBwYXRoID0gb3V0cHV0X2RpciAvIGYie2V4cF9pZH0uanNvbiINCiAgICBwYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9MiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBkZWZhdWx0PWpzb25fZGVmYXVsdCksIGVuY29kaW5nPSJ1dGYtOCIpDQogICAgcmV0dXJuIHN0cihwYXRoKQ0KDQoNCmRlZiBsb2FkX3Jlc3VsdHMocmVzdWx0c19kaXI6IHN0ciA9ICIuLi9yZXN1bHRzIikgLT4gbGlzdFtkaWN0XToNCiAgICAiIiLEkOG7jWMgbeG7jWkgZmlsZSAqLmpzb24gdHJvbmcgcmVzdWx0c19kaXIsIHRy4bqjIHbhu4EgZGFuaCBzw6FjaCBkaWN0IChz4bqvcCB0aGVvIGV4cF9pZCkuIiIiDQogICAgZGlyZWN0b3J5ID0gUGF0aChyZXN1bHRzX2RpcikNCiAgICBpZiBub3QgZGlyZWN0b3J5LmV4aXN0cygpOg0KICAgICAgICByZXR1cm4gW10NCiAgICByZXN1bHRzID0gW2pzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpIGZvciBwYXRoIGluIHNvcnRlZChkaXJlY3RvcnkuZ2xvYigiKi5qc29uIikpXQ0KICAgIHJldHVybiBzb3J0ZWQocmVzdWx0cywga2V5PWxhbWJkYSBpdGVtOiBpdGVtWyJjZmciXS5nZXQoImV4cF9pZCIsICIiKSkNCg0KDQpkZWYgdG9fcm93KHJlc3VsdDogZGljdCwgZXZhbF9zY29yZXM6IGRpY3QgfCBOb25lID0gTm9uZSwgbm90ZXM6IHN0ciA9ICIiKSAtPiBkaWN0Og0KICAgICIiIkJp4bq/biBt4buZdCBr4bq/dCBxdeG6oyB0aMOgbmggbeG7mXQgZMOybmcgY+G7p2EgYuG6o25nOiBn4buZcCBjZmcgKyBzdW1tYXJ5ICgrIGV2YWxfYWNjLCBldmFsX21hY3JvX2YxIG7hur91IGPDsykNCiAgICArIGZpZ3VyZV9maWxlID0gZiJmaWd1cmVzL3tleHBfaWR9LnBuZyIuIEtob8OhIHBo4bqjaSB0csO5bmcgdMOqbiBj4buZdCDhu58gxJHhuqd1IGZpbGUuDQogICAgQ2jhu4kgdHJ1eeG7gW4gZXZhbF9zY29yZXMgY2hvIGJhc2VsaW5lIHbDoCBj4bqldSBow6xuaCBjdeG7kWkgY8O5bmcuIiIiDQogICAgY2ZnID0gcmVzdWx0WyJjZmciXQ0KICAgIHN1bW1hcnkgPSByZXN1bHRbInN1bW1hcnkiXQ0KICAgIHJvdyA9IHsqKmNmZywgKipzdW1tYXJ5fQ0KICAgIHJvd1siaGlkZGVuIl0gPSAiLSIuam9pbihzdHIod2lkdGgpIGZvciB3aWR0aCBpbiBjZmcuZ2V0KCJoaWRkZW4iLCAoKSkpDQogICAgcm93WyJldmFsX2FjYyJdID0gTm9uZSBpZiBldmFsX3Njb3JlcyBpcyBOb25lIGVsc2UgZXZhbF9zY29yZXMuZ2V0KCJhY2N1cmFjeSIpDQogICAgcm93WyJldmFsX21hY3JvX2YxIl0gPSBOb25lIGlmIGV2YWxfc2NvcmVzIGlzIE5vbmUgZWxzZSBldmFsX3Njb3Jlcy5nZXQoIm1hY3JvX2YxIikNCiAgICByb3dbImZpZ3VyZV9maWxlIl0gPSBmImZpZ3VyZXMve2NmZ1snZXhwX2lkJ119LnBuZyINCiAgICByb3dbIm5vdGVzIl0gPSBub3RlcyBvciBjZmcuZ2V0KCJub3RlcyIsICIiKQ0KICAgIHJldHVybiByb3cNCg0KDQpkZWYgd3JpdGVfeGxzeChyb3dzOiBsaXN0W2RpY3RdLCB0ZW1wbGF0ZV9wYXRoOiBzdHIsIG91dF9wYXRoOiBzdHIsDQogICAgICAgICAgICAgICBzZWVkX2V4cF9pZHM6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lLA0KICAgICAgICAgICAgICAgc3VtbWFyeV9ub3RlczogZGljdFtzdHIsIHN0cl0gfCBOb25lID0gTm9uZSkgLT4gTm9uZToNCiAgICAiIiLEkGnhu4FuIGPDoWMgZMOybmcgdsOgbyBzaGVldCAiRXhwZXJpbWVudHMiIGPhu6dhIG3huqt1LCB04burIGTDsm5nIDIgdHLhu58geHXhu5FuZywgcuG7k2kgbMawdSB0aMOgbmggb3V0X3BhdGguDQoNCiAgICBDw6FjIGLGsOG7m2MgKG9wZW5weXhsKToNCiAgICAgIDEuIHdiID0gb3BlbnB5eGwubG9hZF93b3JrYm9vayh0ZW1wbGF0ZV9wYXRoKSAgICMgS0jDlE5HIGTDuW5nIGRhdGFfb25seT1UcnVlIChz4bq9IG3huqV0IGPDtG5nIHRo4bupYykNCiAgICAgIDIuIHdzID0gd2JbIkV4cGVyaW1lbnRzIl07IMSR4buNYyB0acOqdSDEkeG7gSBkw7JuZyAxIMSR4buDIGJp4bq/dCBj4buZdCBuw6BvIOG7qW5nIHbhu5tpIGtob8OhIG7DoG8NCiAgICAgIDMuIHbhu5tpIG3hu5dpIHJvdzogZ2hpIGdpw6EgdHLhu4sgdsOgbyDEkcO6bmcgY+G7mXQ7IELhu44gUVVBIGPDoWMgY+G7mXQgY8O0bmcgdGjhu6ljIChzdGVwMF9nYXBfdnNfbG5DLCBnYXBfdmFsX21pbnVzX3RyYWluLA0KICAgICAgICAgZGVsdGFfdmFsX2YxX3ZzX2Jhc2UsIGJleW9uZF9ub2lzZSkNCiAgICAgIDQuIHdiLnNhdmUob3V0X3BhdGgpDQogICAgU2F1IGtoaSBsxrB1LCBt4bufIGZpbGUgYuG6sW5nIEV4Y2VsL0xpYnJlT2ZmaWNlIMSR4buDIGPDoWMgY8O0bmcgdGjhu6ljIHTDrW5oIGzhuqFpLg0KICAgICIiIg0KICAgIGltcG9ydCBvcGVucHl4bA0KDQogICAgd29ya2Jvb2sgPSBvcGVucHl4bC5sb2FkX3dvcmtib29rKHRlbXBsYXRlX3BhdGgpDQogICAgd29ya3NoZWV0ID0gd29ya2Jvb2tbIkV4cGVyaW1lbnRzIl0NCiAgICBoZWFkZXJzID0gW2NlbGwudmFsdWUgZm9yIGNlbGwgaW4gd29ya3NoZWV0WzFdXQ0KICAgIGZvcm11bGFfY29sdW1ucyA9IHsic3RlcDBfZ2FwX3ZzX2xuQyIsICJnYXBfdmFsX21pbnVzX3RyYWluIiwgImRlbHRhX3ZhbF9mMV92c19iYXNlIiwgImJleW9uZF9ub2lzZSJ9DQogICAgZm9yIHJvd19pbmRleCwgcm93IGluIGVudW1lcmF0ZShyb3dzLCBzdGFydD0yKToNCiAgICAgICAgZm9yIGNvbHVtbl9pbmRleCwgaGVhZGVyIGluIGVudW1lcmF0ZShoZWFkZXJzLCBzdGFydD0xKToNCiAgICAgICAgICAgIGlmIGhlYWRlciBpbiBmb3JtdWxhX2NvbHVtbnM6DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIGlmIGhlYWRlciBpbiByb3c6DQogICAgICAgICAgICAgICAgdmFsdWUgPSByb3dbaGVhZGVyXQ0KICAgICAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIChkaWN0LCBsaXN0LCB0dXBsZSkpOg0KICAgICAgICAgICAgICAgICAgICB2YWx1ZSA9IGpzb24uZHVtcHModmFsdWUsIGVuc3VyZV9hc2NpaT1GYWxzZSkNCiAgICAgICAgICAgICAgICB3b3Jrc2hlZXQuY2VsbChyb3c9cm93X2luZGV4LCBjb2x1bW49Y29sdW1uX2luZGV4LCB2YWx1ZT12YWx1ZSkNCg0KICAgICMgVGhlIHRlbXBsYXRlJ3MgU2VlZHMgYW5kIFN1bW1hcnkgc2hlZXRzIGNvbnRhaW4gZm9ybXVsYXMgYW5kIHRoZWlyIG93bg0KICAgICMgaGVhZGVycy4gRmlsbCBvbmx5IHRoZSBpbnB1dCBjZWxscyBzbyB0aG9zZSBmb3JtdWxhcyByZW1haW4gaW50YWN0Lg0KICAgIGlmIHNlZWRfZXhwX2lkcyBpcyBub3QgTm9uZSBhbmQgIlNlZWRzIiBpbiB3b3JrYm9vay5zaGVldG5hbWVzOg0KICAgICAgICBzZWVkcyA9IHdvcmtib29rWyJTZWVkcyJdDQogICAgICAgIHNlZWRfaGVhZGVycyA9IFtjZWxsLnZhbHVlIGZvciBjZWxsIGluIHNlZWRzWzFdXQ0KICAgICAgICBleHBfaWRfY29sdW1uID0gbmV4dCgNCiAgICAgICAgICAgIGluZGV4IGZvciBpbmRleCwgaGVhZGVyIGluIGVudW1lcmF0ZShzZWVkX2hlYWRlcnMsIHN0YXJ0PTEpDQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGhlYWRlciwgc3RyKSBhbmQgaGVhZGVyLnN0YXJ0c3dpdGgoImV4cF9pZCIpDQogICAgICAgICkNCiAgICAgICAgZm9yIG9mZnNldCwgZXhwX2lkIGluIGVudW1lcmF0ZShzZWVkX2V4cF9pZHMsIHN0YXJ0PTIpOg0KICAgICAgICAgICAgc2VlZHMuY2VsbChyb3c9b2Zmc2V0LCBjb2x1bW49ZXhwX2lkX2NvbHVtbiwgdmFsdWU9ZXhwX2lkKQ0KDQogICAgaWYgc3VtbWFyeV9ub3RlcyBhbmQgIlN1bW1hcnkiIGluIHdvcmtib29rLnNoZWV0bmFtZXM6DQogICAgICAgIHN1bW1hcnkgPSB3b3JrYm9va1siU3VtbWFyeSJdDQogICAgICAgIHN1bW1hcnlfaGVhZGVycyA9IFtjZWxsLnZhbHVlIGZvciBjZWxsIGluIHN1bW1hcnlbMV1dDQogICAgICAgIGdyb3VwX2NvbHVtbiA9IHN1bW1hcnlfaGVhZGVycy5pbmRleCgiZ3JvdXAiKSArIDENCiAgICAgICAgbm90ZXNfY29sdW1uID0gbmV4dCgNCiAgICAgICAgICAgIGluZGV4IGZvciBpbmRleCwgaGVhZGVyIGluIGVudW1lcmF0ZShzdW1tYXJ5X2hlYWRlcnMsIHN0YXJ0PTEpDQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGhlYWRlciwgc3RyKSBhbmQgaGVhZGVyLnN0YXJ0c3dpdGgoIm5o4bqtbiB4w6l0IikNCiAgICAgICAgKQ0KICAgICAgICBmb3Igcm93X2luZGV4IGluIHJhbmdlKDIsIHN1bW1hcnkubWF4X3JvdyArIDEpOg0KICAgICAgICAgICAgZ3JvdXAgPSBzdW1tYXJ5LmNlbGwocm93PXJvd19pbmRleCwgY29sdW1uPWdyb3VwX2NvbHVtbikudmFsdWUNCiAgICAgICAgICAgIGlmIGdyb3VwIGluIHN1bW1hcnlfbm90ZXM6DQogICAgICAgICAgICAgICAgc3VtbWFyeS5jZWxsKHJvdz1yb3dfaW5kZXgsIGNvbHVtbj1ub3Rlc19jb2x1bW4sIHZhbHVlPXN1bW1hcnlfbm90ZXNbZ3JvdXBdKQ0KICAgIG91dHB1dCA9IFBhdGgob3V0X3BhdGgpDQogICAgb3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgd29ya2Jvb2suc2F2ZShvdXRwdXQpDQo="}
    _code_dir = _repo_found / 'code'
    _code_dir.mkdir(parents=True,exist_ok=True)
    for _name,_encoded in _bundle.items(): (_code_dir/_name).write_bytes(base64.b64decode(_encoded))
    os.environ['LAB_REPO_ROOT'] = str(_repo_found.resolve())
    print('Kaggle repo ready:',_repo_found)
elif _repo_found is not None:
    print('Using existing lab repo:',_repo_found)
else:
    print('No Kaggle clone needed; configure REPO_ROOT_OVERRIDE if running elsewhere.')


In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt
REPO_ROOT_OVERRIDE = os.environ.get('LAB_REPO_ROOT')
if REPO_ROOT_OVERRIDE is None:
    _kaggle_repo = Path('/kaggle/working/K4-Track4-Day1-Neuralnetwork')
    if (_kaggle_repo / 'scripts' / 'split_data.py').exists(): REPO_ROOT_OVERRIDE = str(_kaggle_repo)
_cwd = Path.cwd().resolve()
_candidates = [_cwd,_cwd.parent,_cwd.parent.parent]
REPO_ROOT = (Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else next((p for p in _candidates if (p/'scripts'/'split_data.py').exists()),None))
if REPO_ROOT is None: raise FileNotFoundError('Cannot find repository root containing scripts/split_data.py')
REPO_ROOT = str(REPO_ROOT)
OUT_DIR_OVERRIDE = os.environ.get('LAB_OUT_DIR')
if OUT_DIR_OVERRIDE: OUT_DIR = str(Path(OUT_DIR_OVERRIDE).resolve())
elif _cwd.name == 'code': OUT_DIR = str(_cwd.parent)
else: OUT_DIR = str(Path(REPO_ROOT)/'submission_working')
CODE_DIR = _cwd if _cwd.name == 'code' else Path(REPO_ROOT)/'code'
if os.name == 'nt':
    _base_site = Path(sys.prefix).parent.parent/'Lib'/'site-packages'
    if _base_site.exists() and str(_base_site) not in sys.path: sys.path.append(str(_base_site))
sys.path.insert(0,str(CODE_DIR))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}; device: {device}')
if device.type != 'cuda': raise RuntimeError('Kaggle GPU accelerator is not enabled; select a GPU in Notebook settings before Run All.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
os.makedirs(Path(OUT_DIR)/'figures',exist_ok=True); os.makedirs(Path(OUT_DIR)/'results',exist_ok=True)
from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
repo_root = Path(REPO_ROOT)
subprocess.run([sys.executable, str(repo_root / 'scripts' / 'split_data.py')], cwd=REPO_ROOT, check=True)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=str(repo_root / 'data' / 'processed'))
print('Shapes:', {k: tuple(data[k].shape) for k in ('X_tr', 'y_tr', 'X_val', 'y_val', 'X_eval', 'y_eval')})
for name in ('y_tr', 'y_val', 'y_eval'):
    counts = torch.bincount(data[name].cpu(), minlength=7).numpy()
    print(f'{name} class proportions (%):', (100 * counts / counts.sum()).round(2))
print('Eval row_id count/unique:', len(data['eval_row_id']), len(np.unique(data['eval_row_id'])))
# Demonstrate that a final short batch is kept: 1,025 samples -> 512 + 512 + 1.
batch_sizes = [len(xb) for xb, _ in iterate_batches(data['X_tr'][:1025], data['y_tr'][:1025], batch_size=512, shuffle=False)]
print('Example batch sizes:', batch_sizes)
assert tuple(data['X_tr'].shape) == (371847, 54) and tuple(data['X_val'].shape) == (92962, 54)
assert tuple(data['X_eval'].shape) == (116203, 54)
assert np.allclose(data['X_tr'][:, :10].mean(dim=0).cpu().numpy(), 0, atol=2e-4)
assert np.allclose(data['X_tr'][:, :10].std(dim=0, unbiased=False).cpu().numpy(), 1, atol=2e-3)
assert batch_sizes == [512, 512, 1]

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# M-base: shape và số tham số phải khớp quy định.
torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(42)
model = MLP(hidden=(256, 128), dropout=0.0, init='he').to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47879
x_demo = torch.randn(8, 54, device=device)
h = x_demo
for layer in model.net:
    h = layer(h)
    print(f'{layer.__class__.__name__}: {tuple(h.shape)}')
assert h.shape == (8, 7)

# Loss bước 0 trên toàn bộ validation, trước khi có bất kỳ cập nhật nào.
import torch.nn.functional as F
model.eval()
step0_loss_sum = 0.0
with torch.no_grad():
    for start in range(0, len(data['X_val']), 8192):
        xb = data['X_val'][start:start + 8192]
        yb = data['y_val'][start:start + 8192]
        step0_loss_sum += F.cross_entropy(model(xb), yb, reduction='sum').item()
step0_loss = step0_loss_sum / len(data['y_val'])
print(f'Initial validation CE loss: {step0_loss:.6f}; ln(7): {np.log(7):.6f}')
print('Activation std after each ReLU (initial M-base):', activation_stats(model, data['X_val'][:1024]))

# Một lần backward: mọi tensor tham số cần nhận gradient khác None và khác 0.
model.train()
model.zero_grad(set_to_none=True)
loss_for_grad = F.cross_entropy(model(data['X_tr'][:128]), data['y_tr'][:128])
loss_for_grad.backward()
grad_norms = {}
for name, parameter in model.named_parameters():
    assert parameter.grad is not None, f'{name} has no gradient'
    grad_norms[name] = float(parameter.grad.norm().item())
    print(f'{name}: grad norm = {grad_norms[name]:.6g}')
    assert grad_norms[name] > 0, f'{name} has a zero gradient'

# Overfit 20 training examples with dropout off; this checks the model and update path.
torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(42)
small_indices = torch.randperm(len(data['X_tr']), device=device)[:20]
x_small, y_small = data['X_tr'][small_indices], data['y_tr'][small_indices]
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init='he').to(device)
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.01)
overfit_losses = []
overfit_model.train()
for _ in range(500):
    overfit_optimizer.zero_grad(set_to_none=True)
    small_loss = F.cross_entropy(overfit_model(x_small), y_small)
    small_loss.backward()
    overfit_optimizer.step()
    overfit_losses.append(float(small_loss.item()))
overfit_model.eval()
with torch.no_grad():
    final_small_loss = F.cross_entropy(overfit_model(x_small), y_small).item()
    small_accuracy = (overfit_model(x_small).argmax(dim=1) == y_small).float().mean().item()
print(f'20-example overfit: final loss = {final_small_loss:.6f}; accuracy = {small_accuracy:.1%}')
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(overfit_losses) + 1), overfit_losses)
plt.xlabel('Update step'); plt.ylabel('Cross-entropy loss')
plt.title('Part 1: overfit check on 20 training examples'); plt.grid(alpha=0.25)
plt.tight_layout(); plt.show()
assert small_accuracy == 1.0 and final_small_loss < 0.05

**Nhận xét Part 1 (điền sau khi chạy):** ghi loss bước 0 đo được và độ lệch so với ln 7 ≈ 1,946; nhận xét đường loss khi quá khớp 20 mẫu, accuracy cuối; và xác nhận mọi tham số có gradient khác 0. Nếu phép thử không đạt, hãy chẩn đoán trước khi chạy huấn luyện dài.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# Chọn learning rate chỉ bằng validation; mọi ứng viên giữ nguyên các thiết lập baseline khác.
lr_candidates = [0.01, 0.03, 0.1]
lr_sweep_results = []
for candidate_lr in lr_candidates:
    lr_tag = str(candidate_lr).replace('.', 'p')
    candidate_cfg = {**DEFAULT_CFG, 'lr': candidate_lr, 'seed': 1,
                    'exp_id': f'lr-sweep-{lr_tag}', 'group': 'lr_search',
                    'description': f'Baseline learning-rate selection: lr={candidate_lr}'}
    candidate_result = run_experiment(candidate_cfg, data)
    lr_sweep_results.append(candidate_result)
    print(candidate_cfg['exp_id'], candidate_result['summary'])

best_lr_index = max(
    range(len(lr_sweep_results)),
    key=lambda i: (not lr_sweep_results[i]['summary']['diverged'],
                   lr_sweep_results[i]['summary']['val_macro_f1']),
)
selected_lr = lr_candidates[best_lr_index]
print(f'Validation-selected baseline lr: {selected_lr}')

# Lưu mọi ứng viên. Gán lần chạy ứng viên tốt nhất thành base-s1 để dùng làm seed baseline thứ nhất.
for i, result in enumerate(lr_sweep_results):
    if i == best_lr_index:
        result['cfg'] = {**result['cfg'], 'exp_id': 'base-s1', 'group': 'baseline',
                         'description': f'M-base baseline, selected lr={selected_lr}, seed=1'}
    save_result(result, f'{OUT_DIR}/results')
    plot_run(result, f"{OUT_DIR}/figures/{result['cfg']['exp_id']}.png")
plot_compare(lr_sweep_results, 'val_macro_f1', f'{OUT_DIR}/figures/compare_lr_search.png',
             title='Baseline learning-rate selection on validation')

# Hai seed bổ sung đo độ dao động; cùng split, lr, số epoch và mọi cấu hình khác.
baseline_results = [lr_sweep_results[best_lr_index]]
for seed in (2, 3):
    seed_cfg = {**DEFAULT_CFG, 'lr': selected_lr, 'seed': seed,
                'exp_id': f'base-s{seed}', 'group': 'baseline',
                'description': f'M-base baseline, selected lr={selected_lr}, seed={seed}'}
    seed_result = run_experiment(seed_cfg, data)
    baseline_results.append(seed_result)
    save_result(seed_result, f'{OUT_DIR}/results')
    plot_run(seed_result, f"{OUT_DIR}/figures/{seed_cfg['exp_id']}.png")
    print(seed_cfg['exp_id'], seed_result['summary'])

baseline_f1 = np.array([r['summary']['val_macro_f1'] for r in baseline_results])
baseline_acc = np.array([r['summary']['val_acc'] for r in baseline_results])
f1_std = baseline_f1.std(ddof=1)
acc_std = baseline_acc.std(ddof=1)
print(f'Baseline val macro-F1: {baseline_f1.mean():.4f} ± {f1_std:.4f}; 2σ = {2*f1_std:.4f}')
print(f'Baseline val accuracy: {baseline_acc.mean():.4f} ± {acc_std:.4f}')
plot_compare(baseline_results, 'val_macro_f1', f'{OUT_DIR}/figures/compare_baseline_seeds.png',
             title='Baseline validation macro-F1 across seeds')

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Kế hoạch thí nghiệm và dự đoán (ghi trước khi chạy)
Mọi lần chạy dùng cùng phép tách train/validation, seed 1 và 20 epoch như baseline; chỉ đổi yếu tố nêu dưới đây. Các trial Adam là quét learning rate trong chủ đề optimizer, rồi chọn bằng validation.

| Chủ đề / exp_id | Thay đổi | Dự đoán trước khi chạy |
|---|---|---|
| Loss `loss-mse` | CE → MSE logits/one-hot | MSE hồi quy trực tiếp logits về one-hot và lấy trung bình qua 7 đầu ra, nên mục tiêu/thang gradient khác CE; metric có thể hội tụ chậm hơn. So macro-F1/accuracy và tốc độ, không so độ lớn hai loss. |
| Optimizer `opt-adam-*` | SGD+momentum → Adam; thử 3 lr | Adam có thể đạt metric validation tốt sớm hơn nhờ bước thích nghi theo tham số; kết luận chỉ sau khi so lr tốt nhất của từng optimizer. |
| Hyperparameter `batch-2048` | batch 512 → 2048 | Cùng 20 epoch nhưng ít bước cập nhật hơn, nên có thể học chậm hơn; mỗi epoch có thể nhanh hơn. |
| Dropout `drop-0p3` | dropout 0 → 0.3 | Nếu baseline có khoảng cách train–val, dropout có thể giảm overfit; nếu chưa overfit, metric có thể giảm. |
| Clipping `clip-highlr-*` | cắt / không cắt ở cùng lr cao | Với lr = 10× baseline, clipping theo ngưỡng rút ra từ grad_norm p95 có thể giảm dao động hoặc ngăn phân kỳ. So sánh hai lần chạy cùng seed/config, chỉ khác clip. |
| Mixed precision `amp-fp16` | FP32 → FP16 autocast | Kỳ vọng metric gần FP32 và giảm thời gian/bộ nhớ GPU; chỉ gọi nhanh hơn nếu phép đo xác nhận. Chỉ chạy khi có CUDA. |
| Initialization `init-xavier` | He → Xavier normal | Xavier có thể giữ phương sai kích hoạt khác He với ReLU; quan sát loss bước 0, metric và gradient. |

Clipping dùng `c` bằng trung vị p95 grad_norm theo epoch từ baseline. Cặp high-lr giữ cùng seed và learning rate để cô lập tác dụng của clipping.

In [ ]:
part3_results = {}
baseline_f1_mean = float(baseline_f1.mean())
baseline_noise_2sigma = float(2 * f1_std)
base_cfg = {**DEFAULT_CFG, 'lr': selected_lr, 'seed': 1}

def run_part3(exp_cfg, prediction):
    print(f"\nPREDICTION before {exp_cfg['exp_id']}: {prediction}")
    result = run_experiment(exp_cfg, data)
    part3_results[exp_cfg['exp_id']] = result
    save_result(result, f'{OUT_DIR}/results')
    plot_run(result, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    score = result['summary']['val_macro_f1']
    delta = score - baseline_f1_mean
    if exp_cfg['group'] == 'clipping':
        print('Fraction of batches clipped:', float(np.mean(result['history']['clip_fraction'])))
    if exp_cfg['group'] == 'init':
        print('Initialization step-0 loss:', result['summary']['step0_loss'])
        print('Activation std after each ReLU:', result['summary']['activation_std_after_relu_step0'])
    if exp_cfg['group'] == 'amp':
        print('AMP timing/memory:', result['summary']['time_per_epoch_s'], 's/epoch;', result['summary']['peak_mem_MB'], 'MiB peak')
    print(f"AFTER {exp_cfg['exp_id']}: val_macro_f1={score:.4f}; baseline mean={baseline_f1_mean:.4f}; "
          f"delta={delta:+.4f}; baseline 2σ={baseline_noise_2sigma:.4f}; diverged={result['summary']['diverged']}")
    return result

# Loss: only the loss function changes from the selected baseline.
loss_mse = run_part3(
    {**base_cfg, 'exp_id': 'loss-mse', 'group': 'loss', 'loss': 'mse',
     'description': 'MSE on logits vs one-hot labels',
     'notes': 'MSE averages over batch and class dimensions; compare classification metrics, not loss values.'},
    'MSE may improve more slowly than CE because its error gradient can saturate for confidently wrong predictions.'
)

# Optimizer: tune Adam over three validation-only learning rates; other settings stay at baseline.
adam_results = []
for adam_lr in (0.0003, 0.001, 0.003):
    adam_tag = str(adam_lr).replace('.', 'p')
    adam_result = run_part3(
        {**base_cfg, 'exp_id': f'opt-adam-lr{adam_tag}', 'group': 'optimizer',
         'optimizer': 'adam', 'lr': adam_lr,
         'description': f'Adam validation learning-rate trial, lr={adam_lr}',
         'notes': 'Adam lr selected from a three-value validation sweep.'},
        'Adaptive per-parameter steps may improve early optimization; the middle learning rate may balance speed and stability.'
    )
    adam_results.append(adam_result)
best_adam_result = max(adam_results, key=lambda r: (not r['summary']['diverged'], r['summary']['val_macro_f1']))

# Hyperparameter: larger batch, fixed learning rate and epoch count.
batch_result = run_part3(
    {**base_cfg, 'exp_id': 'batch-2048', 'group': 'hparam', 'batch': 2048,
     'description': 'Batch size 2048 vs baseline 512',
     'notes': 'Same epoch count means fewer optimizer updates; compare time per epoch too.'},
    'A larger batch may smooth gradients and shorten epochs, but it makes fewer updates over the same 20 epochs.'
)

# Dropout: the only change is the hidden-layer dropout probability.
dropout_result = run_part3(
    {**base_cfg, 'exp_id': 'drop-0p3', 'group': 'dropout', 'dropout': 0.3,
     'description': 'Hidden dropout p=0.3',
     'notes': 'Train loss is measured in eval mode for a fair train/validation gap.'},
    'If the baseline overfits, p=0.3 may narrow the train-validation gap; without overfitting it may lower both metrics.'
)

# Gradient clipping: choose c from baseline pre-clip norms and compare a matched high-lr pair.
baseline_p95 = np.concatenate([r['history']['grad_norm_p95'] for r in baseline_results])
clip_c = float(np.median(baseline_p95[baseline_p95 > 0]))
high_lr = selected_lr * 10
print(f'Clip threshold c={clip_c:.5g} from baseline p95; high learning rate={high_lr:g}')
clip_unclipped = run_part3(
    {**base_cfg, 'exp_id': 'clip-highlr-none', 'group': 'clipping', 'lr': high_lr, 'clip_norm': None,
     'description': f'High learning rate {high_lr:g}, no clipping',
     'notes': 'Matched high-lr control for clip-highlr-c.'},
    'At 10× baseline lr, updates may oscillate or diverge without clipping.'
)
clip_enabled = run_part3(
    {**base_cfg, 'exp_id': 'clip-highlr-c', 'group': 'clipping', 'lr': high_lr, 'clip_norm': clip_c,
     'description': f'High learning rate {high_lr:g}, clip c={clip_c:.5g}',
     'notes': f'Matched high-lr clipping comparison; c from baseline grad_norm p95 median ({clip_c:.5g}).'},
    'Clipping should bound unusually large updates and may stabilize the same high learning rate.'
)

# Mixed precision: run only on CUDA; the FP32 baseline remains the reference.
amp_result = None
if device.type == 'cuda':
    amp_result = run_part3(
        {**base_cfg, 'exp_id': 'amp-fp16', 'group': 'amp', 'precision': 'fp16',
         'description': 'CUDA FP16 autocast with GradScaler',
         'notes': 'Compare measured time and peak memory with FP32 baseline.'},
        'FP16 should preserve validation metrics closely and may reduce GPU memory or epoch time.'
    )
else:
    print('AMP experiment skipped: no CUDA device is available; record this limitation in the report.')

# Initialization: change only He to Xavier normal.
init_result = run_part3(
    {**base_cfg, 'exp_id': 'init-xavier', 'group': 'init', 'init': 'xavier',
     'description': 'Xavier normal initialization',
     'notes': 'Compare step-0 loss, validation metric and gradient history with He baseline.'},
    'Xavier may produce different activation scales from He with ReLU; the effect should appear in step-0 loss and early gradients.'
)

**Đối chiếu sau khi chạy:** dùng các dòng `AFTER` và hình để ghi kết quả đo được, có khớp dự đoán không, cơ chế nào phù hợp với quan sát, và delta macro-F1 có lớn hơn 2σ seed không. Với clipping, xác nhận c có thực sự kích hoạt qua grad_norm trước clip; với AMP, chỉ kết luận nhanh hơn nếu thời gian đo ngắn hơn. Không điền số chưa đo.

In [ ]:
# Ảnh chồng đường cong theo từng nhóm để so sánh trực tiếp.
plot_compare([baseline_results[0], loss_mse], 'val_macro_f1', f'{OUT_DIR}/figures/compare_loss.png', title='Loss comparison')
plot_compare([baseline_results[0], *adam_results], 'val_macro_f1', f'{OUT_DIR}/figures/compare_optimizer.png', title='SGD+momentum baseline vs Adam learning-rate sweep')
plot_compare([baseline_results[0], batch_result], 'val_macro_f1', f'{OUT_DIR}/figures/compare_batch.png', title='Batch-size comparison')
plot_compare([baseline_results[0], dropout_result], 'val_macro_f1', f'{OUT_DIR}/figures/compare_dropout.png', title='Dropout comparison')
plot_compare([baseline_results[0], dropout_result], 'train_loss', f'{OUT_DIR}/figures/compare_dropout_train_loss.png', title='Train loss: dropout comparison')
plot_compare([baseline_results[0], dropout_result], 'val_loss', f'{OUT_DIR}/figures/compare_dropout_val_loss.png', title='Validation loss: dropout comparison')
plot_compare([clip_unclipped, clip_enabled], 'val_macro_f1', f'{OUT_DIR}/figures/compare_clipping.png', title='Matched high-lr clipping comparison')
plot_compare([clip_unclipped, clip_enabled], 'grad_norm', f'{OUT_DIR}/figures/compare_clipping_gradnorm.png', title='Pre-clip gradient norm at high lr')
if amp_result is not None:
    plot_compare([baseline_results[0], amp_result], 'val_macro_f1', f'{OUT_DIR}/figures/compare_amp.png', title='FP32 vs FP16')
    plot_compare([baseline_results[0], amp_result], 'epoch_time_s', f'{OUT_DIR}/figures/compare_amp_time.png', title='Measured epoch time: FP32 vs FP16')
plot_compare([baseline_results[0], init_result], 'val_macro_f1', f'{OUT_DIR}/figures/compare_init.png', title='He vs Xavier initialization')

# Chọn cấu hình cuối cùng CHỈ bằng validation; Part 4 sẽ dùng result_final và cfg_final.
validation_candidates = baseline_results + list(part3_results.values())
valid_candidates = [r for r in validation_candidates if not r['summary']['diverged']]
result_final = max(valid_candidates, key=lambda r: r['summary']['val_macro_f1'])
cfg_final = result_final['cfg']
print('Final configuration selected by validation:', cfg_final['exp_id'], cfg_final)
print('Final validation macro-F1:', result_final['summary']['val_macro_f1'])

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# Evaluate only the baseline and the configuration selected using validation.
import tempfile, shutil
from pathlib import Path
baseline_result = next(r for r in baseline_results if r['cfg']['exp_id']=='base-s1')
def _official_eval(pred_path,json_path):
    run=subprocess.run([sys.executable,str(Path(REPO_ROOT)/'scripts'/'evaluate.py'),'--pred',str(pred_path),'--out',str(json_path)],cwd=REPO_ROOT,check=True,capture_output=True,text=True)
    print(run.stdout)
    return json.loads(Path(json_path).read_text(encoding='utf-8'))
with tempfile.TemporaryDirectory(prefix='baseline_eval_') as _tmp:
    _base_csv=Path(_tmp)/'baseline.csv'; _base_json=Path(_tmp)/'baseline.json'
    final_eval(baseline_result['cfg'],baseline_result,data,str(_base_csv))
    baseline_eval_metrics=_official_eval(_base_csv,_base_json)
    if cfg_final['exp_id']==baseline_result['cfg']['exp_id']:
        shutil.copy2(_base_csv,Path(OUT_DIR)/'predictions_eval.csv')
        eval_metrics=baseline_eval_metrics
        Path(OUT_DIR,'eval_result.json').write_text(json.dumps(eval_metrics,indent=2,ensure_ascii=False),encoding='utf-8')
    else:
        _final_csv=Path(OUT_DIR)/'predictions_eval.csv'; _final_json=Path(OUT_DIR)/'eval_result.json'
        final_eval(cfg_final,result_final,data,str(_final_csv))
        eval_metrics=_official_eval(_final_csv,_final_json)
print('Baseline eval:',{k:baseline_eval_metrics[k] for k in ('n_eval','accuracy','macro_f1')})
print('Final selection (validation only):',cfg_final['exp_id'],result_final['summary']['val_macro_f1'])
print('Final eval:',{k:eval_metrics[k] for k in ('n_eval','accuracy','macro_f1')})


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
per_class=pd.DataFrame(eval_metrics['per_class'])
display(per_class[['cls','support','precision','recall','f1']].round(4))
confusion_df=pd.DataFrame(eval_metrics['confusion_matrix'],index=range(7),columns=range(7))
confusion_df.index.name='actual / predicted'
display(confusion_df)
hardest=min(eval_metrics['per_class'],key=lambda row:row['f1'])
class_id=int(hardest['cls']); _confused=list(eval_metrics['confusion_matrix'][class_id]); _confused[class_id]=0
most_confused=int(np.argmax(_confused)); confused_count=int(_confused[most_confused])
error_analysis_md=(f"On eval, class **{class_id}** has the lowest F1 ({hardest['f1']:.4f}; support {hardest['support']:,}) "
f"and is most often confused with class **{most_confused}** ({confused_count:,} examples). Its low support makes the per-class estimate sensitive to individual mistakes; the confusion matrix shows observed errors, not their cause.")
display(Markdown(error_analysis_md))


**Phân tích lỗi:** bảng per-class, ma trận nhầm lẫn và nhận xét phía trên được dựng trực tiếp từ `eval_result.json`.

In [ ]:
all_results=load_results(f'{OUT_DIR}/results')
eval_by_id={baseline_result['cfg']['exp_id']:baseline_eval_metrics}; eval_by_id[cfg_final['exp_id']]=eval_metrics
rows=[to_row(r,eval_scores=eval_by_id.get(r['cfg']['exp_id'])) for r in all_results]
summary_notes={
 'baseline':f"3 seeds; val macro-F1 mean={baseline_f1_mean:.4f}, sample SD={f1_std:.4f}, 2SD={baseline_noise_2sigma:.4f}.",
 'loss':f"MSE trial val macro-F1={part3_results['loss-mse']['summary']['val_macro_f1']:.4f}; compare classification metrics, not loss scales.",
 'optimizer':f"Best Adam: {best_adam_result['cfg']['exp_id']}, val macro-F1={best_adam_result['summary']['val_macro_f1']:.4f}; lr selected on val.",
 'hparam':f"batch-2048 val macro-F1={batch_result['summary']['val_macro_f1']:.4f}; batch changes updates/epoch.",
 'dropout':f"drop-0p3 val macro-F1={dropout_result['summary']['val_macro_f1']:.4f}; compare train/val gap.",
 'clipping':f"High-lr clipped fraction={np.mean(clip_enabled['history']['clip_fraction']):.3f}; matched unclipped run included.",
 'amp':(f"FP16 val macro-F1={amp_result['summary']['val_macro_f1']:.4f}; {amp_result['summary']['time_per_epoch_s']:.2f}s/epoch, {amp_result['summary']['peak_mem_MB']:.1f} MiB peak.") if amp_result else 'AMP unavailable on this runtime.',
 'init':f"Xavier val macro-F1={init_result['summary']['val_macro_f1']:.4f}; compare activation scale and step-0 loss with He.",
}
write_xlsx(rows,str(Path(REPO_ROOT)/'templates'/'experiment_table_template.xlsx'),str(Path(OUT_DIR)/'experiments.xlsx'),seed_exp_ids=[r['cfg']['exp_id'] for r in baseline_results],summary_notes=summary_notes)
print(f"Workbook saved with {len(rows)} experiment rows; eval columns only for baseline/final.")


In [ ]:
from pathlib import Path

def _f(x,n=4): return f"{float(x):.{n}f}"
base_ids=[r['cfg']['exp_id'] for r in baseline_results]
base_acc_mean=float(np.mean([r['summary']['val_acc'] for r in baseline_results])); base_acc_std=float(np.std([r['summary']['val_acc'] for r in baseline_results],ddof=1))
base_gap=baseline_result['summary']['final_val_loss']-baseline_result['summary']['final_train_loss']
exp_lines=[]
for r in all_results:
 c=r['cfg']; s=r['summary']; group=c.get('group','')
 exp_lines.append(f"| `{c['exp_id']}` | {group} | {c.get('lr')} | {_f(s['val_macro_f1'])} | {s['best_epoch']} | {_f(s['time_per_epoch_s'],2)} |")
pc_lines=[f"| {p['cls']} | {p['support']} | {_f(p['precision'])} | {_f(p['recall'])} | {_f(p['f1'])} |" for p in eval_metrics['per_class']]
cm_lines=['| Actual / Predicted | '+' | '.join(map(str,range(7)))+' |','|---|'+'---:|'*7]
for i,row in enumerate(eval_metrics['confusion_matrix']): cm_lines.append(f"| {i} | "+' | '.join(map(str,row))+' |')

report=f'''# Lab Day 1 — Báo cáo thí nghiệm — MSSV chưa cung cấp

## 1. Thiết lập

- Môi trường: Kaggle Notebook; PyTorch {torch.__version__}; thiết bị `{torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU'}`.
- Dữ liệu: Forest CoverType; train 371,847, validation 92,962 (20% stratified từ train, seed 42), eval {len(data['y_eval']):,}. Chỉ 10 biến liên tục được chuẩn hoá theo thống kê train; 44 one-hot giữ nguyên.
- Model M-base `54→256→128→7`, {count_params(model):,} tham số. Baseline: CE, SGD+momentum 0.9, lr={selected_lr}, batch 512, 20 epoch, He, dropout 0, không clipping, FP32.
- Chủ đề đã chạy: loss, optimizer, batch-size, dropout, clipping, FP16, init.

## 2. Kiểm tra ban đầu và độ nhiễu

| Kiểm tra | Kết quả |
|---|---|
| Tham số / logits | {count_params(model):,} / (8, 7) |
| Loss bước 0 / ln(7) | {_f(step0_loss,6)} / {_f(np.log(7),6)} |
| Quá khớp 20 mẫu | loss {_f(final_small_loss,6)}, accuracy {small_accuracy:.1%} sau 500 bước |
| Gradient mọi tham số | Có, mọi tham số nhận gradient khác 0 |
| Baseline seeds | {', '.join('`'+s+'`' for s in base_ids)} |
| Baseline val accuracy | {base_acc_mean:.4f} ± {base_acc_std:.4f} (sample SD) |
| Baseline val macro-F1 | {baseline_f1_mean:.4f} ± {f1_std:.4f} (sample SD) |

Ngưỡng nhiễu tham chiếu 2σ={baseline_noise_2sigma:.4f} macro-F1 từ ba seed; ước lượng này còn bất định. Chênh lệch nhỏ hơn ngưỡng không đủ để khẳng định cải thiện. Kiểm tra 20 mẫu đạt loss thấp và accuracy 100%, xác nhận đường forward/backward và optimizer hoạt động trên batch nhỏ.

## 3. Kết quả theo chủ đề

Dự đoán trước từng lần chạy được ghi trong notebook trước khi gọi hàm huấn luyện. Mọi metric lựa chọn dưới đây là validation; bảng đầy đủ và ảnh riêng theo `exp_id` nằm trong workbook và `figures/`.

| exp_id | Nhóm | lr | Val macro-F1 | Best epoch | s/epoch |
|---|---|---:|---:|---:|---:|
{chr(10).join(exp_lines)}

- **Loss CE/MSE.** `loss-mse` đạt {_f(loss_mse['summary']['val_macro_f1'])}, chênh {loss_mse['summary']['val_macro_f1']-baseline_f1_mean:+.4f} so với baseline mean; 2σ={baseline_noise_2sigma:.4f}. MSE lấy bình phương sai số logits–one-hot và trung bình qua lớp, nên gradient/thang loss khác CE. So macro-F1/accuracy và đường cong, không so trực tiếp độ lớn loss. [So sánh](figures/compare_loss.png).
- **Optimizer.** Baseline SGD (lr chọn bằng validation) đạt {_f(baseline_result['summary']['val_macro_f1'])}. Adam tốt nhất `{best_adam_result['cfg']['exp_id']}` (lr={best_adam_result['cfg']['lr']}) đạt {_f(best_adam_result['summary']['val_macro_f1'])}, best epoch {best_adam_result['summary']['best_epoch']}; chênh với baseline mean {best_adam_result['summary']['val_macro_f1']-baseline_f1_mean:+.4f}. So với 2σ trước khi kết luận; mỗi trial một seed, ba learning rate chỉ là sweep nhỏ. Adam thích nghi bước theo moment gradient, kết quả vẫn nhạy với lr. [Đường cong](figures/compare_optimizer.png).
- **Hyper-parameter (batch).** `batch-2048` đạt {_f(batch_result['summary']['val_macro_f1'])}; cùng epoch nhưng ít cập nhật hơn batch 512. Thời gian/epoch được ghi riêng và không đại diện cùng số bước. [So sánh](figures/compare_batch.png).
- **Dropout.** `drop-0p3` đạt {_f(dropout_result['summary']['val_macro_f1'])}. Baseline train–val loss gap cuối là {base_gap:+.4f} (train metric trên 50k mẫu cố định ở eval mode). Đối chiếu [train loss](figures/compare_dropout_train_loss.png) và [val loss](figures/compare_dropout_val_loss.png); nếu baseline chưa cho thấy gap tăng và dropout giảm metric, lần chạy này chưa ủng hộ dropout.
- **Gradient clipping.** Ngưỡng c={clip_c:.5g} lấy từ median p95 grad norm baseline. Cặp cùng lr cao={high_lr:g}: unclipped diverged={clip_unclipped['summary']['diverged']}, clipped diverged={clip_enabled['summary']['diverged']}; tỷ lệ batch clipped={np.mean(clip_enabled['history']['clip_fraction']):.3f}; chênh macro-F1 clipped–unclipped={clip_enabled['summary']['val_macro_f1']-clip_unclipped['summary']['val_macro_f1']:+.4f}. Norm đo trước clip. [Metric](figures/compare_clipping.png), [grad norm](figures/compare_clipping_gradnorm.png).
- **Mixed precision.** FP32 baseline {_f(baseline_result['summary']['time_per_epoch_s'],2)} s/epoch, {_f(baseline_result['summary']['peak_mem_MB'],1)} MiB; FP16 {_f(amp_result['summary']['time_per_epoch_s'],2)} s/epoch, {_f(amp_result['summary']['peak_mem_MB'],1)} MiB, val macro-F1 {_f(amp_result['summary']['val_macro_f1'])}. Chỉ phép đo này cho biết nhanh/chậm trên runtime GPU hiện tại; autocast overhead và kích thước mạng có thể triệt tiêu lợi ích. [Metric](figures/compare_amp.png), [thời gian](figures/compare_amp_time.png).
- **Khởi tạo.** He step-0 loss {_f(baseline_result['summary']['step0_loss'],6)}, activation std sau ReLU {baseline_result['summary']['activation_std_after_relu_step0']}; Xavier loss {_f(init_result['summary']['step0_loss'],6)}, std {init_result['summary']['activation_std_after_relu_step0']}, macro-F1 {_f(init_result['summary']['val_macro_f1'])}. He dùng variance khoảng 2/fan-in cho ReLU; Xavier cân bằng fan-in/fan-out. [So sánh](figures/compare_init.png).

## 4. Đánh giá cuối trên eval

Cấu hình cuối `{cfg_final['exp_id']}` được chọn bằng val macro-F1 ({_f(result_final['summary']['val_macro_f1'])}) trước khi gọi evaluator. Eval chỉ đo baseline và cấu hình cuối; file JSON là kết quả nguyên bản từ `scripts/evaluate.py`.

| Cấu hình | Seed | Val macro-F1 | Eval macro-F1 | Eval accuracy |
|---|---:|---:|---:|---:|
| Baseline `base-s1` | {baseline_result['cfg']['seed']} | {_f(baseline_result['summary']['val_macro_f1'])} | {_f(baseline_eval_metrics['macro_f1'])} | {_f(baseline_eval_metrics['accuracy'])} |
| Cuối `{cfg_final['exp_id']}` | {cfg_final['seed']} | {_f(result_final['summary']['val_macro_f1'])} | {_f(eval_metrics['macro_f1'])} | {_f(eval_metrics['accuracy'])} |

Eval macro-F1 đổi {eval_metrics['macro_f1']-baseline_eval_metrics['macro_f1']:+.4f} so với baseline. Val 2σ={baseline_noise_2sigma:.4f}; eval mỗi cấu hình chỉ có một seed nên không đo được độ lệch chuẩn eval. Val–eval của cấu hình cuối lệch {eval_metrics['macro_f1']-result_final['summary']['val_macro_f1']:+.4f}; khác biệt split và sampling có thể góp phần.

### 4.1 Phân tích lỗi theo lớp

| Lớp | Support | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
{chr(10).join(pc_lines)}

Lớp thấp nhất là **{class_id}** (F1={_f(hardest['f1'])}, support={hardest['support']:,}), nhầm nhiều nhất với lớp **{most_confused}** ({confused_count:,} mẫu). Support thấp làm metric lớp hiếm nhạy với từng lỗi. Ma trận dưới đây cho biết lỗi quan sát được, không tự giải thích nguyên nhân:

{chr(10).join(cm_lines)}

Có thể kiểm tra giả thuyết bằng phân bố 10 biến số và one-hot giữa hai lớp; một cải tiến tiếp theo là thử class-weighted CE, chỉ chọn trên cùng validation.

## 5. Câu hỏi dẫn dắt

1. Adam tốt hơn SGD trong lần chạy nào? So lr tốt nhất trong sweep: `{best_adam_result['cfg']['exp_id']}` đạt {_f(best_adam_result['summary']['val_macro_f1'])}, SGD baseline {_f(baseline_result['summary']['val_macro_f1'])}; xét 2σ và số seed trước kết luận. Với lr cố định, kết luận có thể đổi vì optimizer có độ nhạy lr khác nhau.
2. Dropout chỉ hữu ích khi validation cho thấy giảm overfit; `drop-0p3` đạt {_f(dropout_result['summary']['val_macro_f1'])} và baseline gap {base_gap:+.4f} nên cần xem cả đường loss.
3. Clipping giới hạn norm trước cập nhật; bằng chứng ở đây là norm trước clip, clip fraction và trạng thái diverged của cặp cùng lr cao, không chỉ nhìn metric cuối.
4. FP16 measured {_f(amp_result['summary']['time_per_epoch_s'],2)} s/epoch so với FP32 {_f(baseline_result['summary']['time_per_epoch_s'],2)}; kết luận phụ thuộc workload và GPU.
5. Zero init khiến neuron cùng lớp có đối xứng gradient nên không học đặc trưng khác nhau. He giữ variance cho ReLU; Xavier cân bằng hai phía và có thể làm activation nhỏ hơn ở mạng ReLU.
6. Sau 2,000 bước loss không giảm: (a) kiểm tra shape/dtype/range nhãn và finite loss trên một batch; (b) thử overfit 20 mẫu khi tắt dropout để tách lỗi model/gradient; (c) xem step-0 loss, gradient norms, zero_grad/optimizer.step và xác nhận tham số đổi. Các bước lần lượt cô lập dữ liệu, đường truyền, và vòng cập nhật.

## 6. Hạn chế

Chỉ ba seed baseline; mỗi chủ đề một seed; sweep lr thưa. Batch-size làm thay đổi số lần cập nhật trong cùng số epoch. Thời gian FP16/FP32 chỉ phản ánh một phiên GPU Kaggle. Lý giải lỗi lớp bằng support/features là giả thuyết cần kiểm tra thêm.

## 7. Phụ lục

- {len(all_results)} lần chạy; `experiments.xlsx`, `predictions_eval.csv`, `eval_result.json`, `figures/`, `results/`, `code/`.
- Tổng thời gian epoch khoảng {sum(r['summary']['time_per_epoch_s']*len(r['history']['epoch']) for r in all_results)/60:.1f} phút, không gồm setup/eval. MSSV chưa được cung cấp, nên thư mục đang là `submission_working`.
'''
Path(OUT_DIR,'REPORT.md').write_text(report,encoding='utf-8')
print('Report written:',Path(OUT_DIR,'REPORT.md'))


In [ ]:
# Assemble the deliverable archive for download from Kaggle.
import shutil, zipfile
from pathlib import Path
submission_code=Path(OUT_DIR)/'code'; submission_code.mkdir(parents=True,exist_ok=True)
for module in Path(CODE_DIR).glob('*.py'): shutil.copy2(module,submission_code/module.name)
print('The executed Kaggle notebook will be downloaded separately and added as code/lab.ipynb after the run.')
archive_path=Path(OUT_DIR).parent/'submission_working.zip'
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for p in Path(OUT_DIR).rglob('*'):
        if p.is_file() and p!=archive_path and '__pycache__' not in p.parts:
            archive.write(p,p.relative_to(Path(OUT_DIR).parent))
print('Staged files:',sorted(str(p.relative_to(OUT_DIR)) for p in Path(OUT_DIR).rglob('*') if p.is_file()))
print('Downloadable archive:',archive_path)
